<a href="https://colab.research.google.com/github/AadishY/Collab/blob/main/Qwen_Image_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🎨 Qwen-Image-2.1 Studio: Text-to-Image, Image Edit, Head Swap & Body Swap
### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

This notebook provides a complete, production-ready suite for **Qwen-Image-2.1** with native ComfyUI integration:
- **Text-to-Image Generation:** Native resolution selection up to 2K (2048×2048) with aspect ratio presets and adaptive token budget.
- **Single-Image Editing & Style-to-Real:** Modify existing images, add/remove elements, or transform anime/sketch/render into photorealistic human characters via `Anything2RealCharacters`. Strictly preserves input aspect ratio by default.
- **Dedicated Head Swap (BFS Head V1.1):** Seamlessly replace head/face while strictly preserving the target scene's aspect ratio, lighting, pose, head angle, micro-expressions, and background from Picture 1.
- **Dedicated Body Swap (BFS Body V1.0):** Transfer full person identity, clothing, and proportions from Image 2 while preserving the target scene's aspect ratio, pose, framing, lighting, and background from Image 1.
- **Interactive LoRA Dropdowns:** Easily select preset LoRAs or enter custom ones via form dropdowns.
- **⚡ Fast / Low-VRAM Aspect-Preserving Mode:** Toggle option to keep the exact target aspect ratio while lowering pixel dimensions for ultra-fast generation and low memory usage.
- **Multi-LoRA Chaining:** Chain multiple LoRAs simultaneously (e.g. Head/Body Swap + 8-Step Turbo acceleration LoRA for 3× faster generations).
- **In-Memory Caching & Auto-Recovery:** Powered by `QwenImage21Cache` with GPU memory optimization, self-healing modules, and OOM guards.


In [ ]:
# @title 1. Environment Setup & Model Synchronization
import os
import sys
import subprocess
import time
from IPython.display import display, HTML

t0 = time.time()

# 1. Package Installation via uv
!pip install -q uv
!apt-get update -qq && apt-get install -y -qq aria2

if not os.path.exists("/content/engine"):
    subprocess.run(["git", "clone", "-q", "https://github.com/comfyanonymous/ComfyUI.git", "/content/engine"])
os.chdir("/content/engine")

!uv pip install --system torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!uv pip install --system -r requirements.txt
!uv pip install --system websocket-client requests Pillow gdown

# 2. Foolproof Download Engine
def foolproof_download(url: str, output_dir: str, exact_filename: str, min_size_mb: float = 100.0):
    os.makedirs(output_dir, exist_ok=True)
    target_file = os.path.join(output_dir, exact_filename)
    aria_meta_file = target_file + ".aria2"

    if os.path.exists(target_file) and not os.path.exists(aria_meta_file):
        size_mb = os.path.getsize(target_file) / (1024 * 1024)
        if size_mb >= min_size_mb:
            return "verified", size_mb

    cmd = [
        "aria2c",
        "--console-log-level=error",
        "-c",
        "-x", "16",
        "-s", "16",
        "-k", "1M",
        "--auto-file-renaming=false",
        "--allow-overwrite=true",
        "--content-disposition=false",
        "-d", output_dir,
        "-o", exact_filename,
        url
    ]

    result = subprocess.run(cmd)
    if result.returncode != 0 or not os.path.exists(target_file):
        raise RuntimeError(f"Download failed for {exact_filename}")

    if os.path.exists(aria_meta_file):
        os.remove(aria_meta_file)

    size_mb = os.path.getsize(target_file) / (1024 * 1024)
    return "downloaded", size_mb

# Base Model Registry (Essential Core Weights)
models_manifest = [
    {
        "name": "Diffusion Model (INT8 convrot)",
        "file": "qwen_image_2.1_int8_convrot.safetensors",
        "dir": "/content/engine/models/diffusion_models",
        "url": "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_int8_convrot.safetensors",
        "min_mb": 500.0
    },
    {
        "name": "Text Encoder (Qwen3-VL 8B INT8)",
        "file": "qwen3vl_8b_int8_convrot.safetensors",
        "dir": "/content/engine/models/text_encoders",
        "url": "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors",
        "min_mb": 500.0
    },
    {
        "name": "VAE (BF16)",
        "file": "qwen_image_2.1_vae_bf16.safetensors",
        "dir": "/content/engine/models/vae",
        "url": "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors",
        "min_mb": 100.0
    }
]

!mkdir -p /content/engine/models/diffusion_models /content/engine/models/text_encoders /content/engine/models/vae /content/engine/models/loras /content/engine/input /content/output

# 3. Write Self-Healing Shared Engine Module
SHARED_UTILS_CODE = 'import os\nimport sys\nimport time\nimport json\nimport math\nimport uuid\nimport random\nimport io\nimport re\nimport subprocess\nimport requests\nimport urllib.request\nimport urllib.parse\nimport websocket\nimport base64\nfrom PIL import Image, ImageOps\nfrom IPython.display import display, HTML\n\n# Host memory conservation flags\nos.environ["TORCH_ALLOW_TF32_CUBLAS_OVERRIDE"] = "1"\nos.environ["CUDA_MODULE_LOADING"] = "LAZY"\nos.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"\n\nserver_address = "127.0.0.1:8188"\nclient_id = str(uuid.uuid4())\nserver_proc = None\n\ndef launch_inference_server():\n    global server_proc\n    engine_dir = "/content/engine" if os.path.exists("/content/engine") else "."\n    server_proc = subprocess.Popen(\n        [\n            "python", "main.py",\n            "--listen", "127.0.0.1",\n            "--port", "8188",\n            "--disable-pinned-memory",\n            "--mmap-torch-files",\n            "--fast",\n            "--cuda-malloc",\n            "--fp16-vae",\n            "--preview-method", "none"\n        ],\n        cwd=engine_dir,\n        stdout=subprocess.PIPE,\n        stderr=subprocess.STDOUT,\n        text=True,\n        bufsize=1\n    )\n    return server_proc\n\ndef ensure_server_alive():\n    global server_address\n    try:\n        r = requests.get(f"http://{server_address}/system_stats", timeout=1.5)\n        if r.status_code == 200:\n            return True\n    except requests.exceptions.RequestException:\n        pass\n\n    print("Backend server is not running. Launching inference server on 127.0.0.1:8188...")\n    launch_inference_server()\n    t_wait = time.time()\n    while time.time() - t_wait < 60:\n        try:\n            r = requests.get(f"http://{server_address}/system_stats", timeout=1.5)\n            if r.status_code == 200:\n                print(f"✓ Backend server initialized and online ({time.time() - t_wait:.1f}s).")\n                return True\n        except requests.exceptions.RequestException:\n            pass\n        time.sleep(1.2)\n    raise RuntimeError("Failed to connect to backend server on 127.0.0.1:8188 after 60s.")\n\ndef purge_backend_memory():\n    try:\n        payload = json.dumps({"unload_models": False, "free_memory": True}).encode("utf-8")\n        req = urllib.request.Request(f"http://{server_address}/free", data=payload, headers={"Content-Type": "application/json"})\n        with urllib.request.urlopen(req, timeout=5) as resp:\n            pass\n    except Exception:\n        pass\n\ndef queue_prompt(prompt_workflow):\n    payload = json.dumps({"prompt": prompt_workflow, "client_id": client_id}).encode("utf-8")\n    req = urllib.request.Request(f"http://{server_address}/prompt", data=payload, headers={"Content-Type": "application/json"})\n    try:\n        with urllib.request.urlopen(req) as resp:\n            res_data = json.loads(resp.read().decode("utf-8"))\n            if "error" in res_data:\n                raise ValueError(f"Prompt validation error: {res_data[\'error\']}")\n            if "node_errors" in res_data and res_data["node_errors"]:\n                raise ValueError(f"Node error in workflow: {res_data[\'node_errors\']}")\n            if "prompt_id" not in res_data:\n                raise ValueError(f"Server rejected prompt without ID: {res_data}")\n            return res_data\n    except urllib.error.HTTPError as e:\n        err_body = e.read().decode("utf-8")\n        raise ValueError(f"Server rejected prompt (HTTP {e.code}): {err_body}")\n\ndef get_image(filename, subfolder, folder_type):\n    params = urllib.parse.urlencode({"filename": filename, "subfolder": subfolder, "type": folder_type})\n    with urllib.request.urlopen(f"http://{server_address}/view?{params}") as response:\n        return response.read()\n\ndef render_dashboard(display_handle, task_name: str, stage: str, node_title: str, progress_pct: int, elapsed_sec: float, seed: int, dims: str, loras_label: str, done: bool = False, error: bool = False):\n    if error:\n        badge_color = "#f85149"\n        status_text = "ERROR"\n        grad = "#da3633"\n    elif done:\n        badge_color = "#3fb950"\n        status_text = "COMPLETED"\n        grad = "#2ea043"\n    else:\n        badge_color = "#58a6ff"\n        status_text = "PROCESSING"\n        grad = "linear-gradient(90deg, #1f6feb, #58a6ff)"\n\n    bar_fill = min(100, max(0, progress_pct))\n\n    html = f"""\n    <div style="font-family: -apple-system, BlinkMacSystemFont, \'Segoe UI\', Roboto, sans-serif; background: #0d1117; color: #e6edf3; border: 1px solid #30363d; border-radius: 10px; padding: 18px; margin: 12px 0; max-width: 820px; box-shadow: 0 8px 24px rgba(0,0,0,0.3);">\n      <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 14px; border-bottom: 1px solid #21262d; padding-bottom: 10px;">\n        <span style="font-size: 13px; font-weight: 600; letter-spacing: 0.5px; color: #58a6ff;">✦ {task_name.upper()}</span>\n        <span style="background: rgba(88, 166, 255, 0.12); color: {badge_color}; border: 1px solid {badge_color}40; padding: 3px 10px; border-radius: 20px; font-size: 11px; font-weight: 600;">{status_text}</span>\n      </div>\n\n      <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; font-size: 12px; margin-bottom: 14px;">\n        <div style="background: #161b22; padding: 8px 12px; border-radius: 6px; border: 1px solid #21262d;">\n          <div style="color: #8b949e; font-size: 10px; text-transform: uppercase;">Seed</div>\n          <div style="color: #f0f6fc; font-weight: 600; font-family: monospace; margin-top: 2px;">{seed}</div>\n        </div>\n        <div style="background: #161b22; padding: 8px 12px; border-radius: 6px; border: 1px solid #21262d;">\n          <div style="color: #8b949e; font-size: 10px; text-transform: uppercase;">Canvas / Dims</div>\n          <div style="color: #f0f6fc; font-weight: 600; margin-top: 2px;">{dims}</div>\n        </div>\n        <div style="background: #161b22; padding: 8px 12px; border-radius: 6px; border: 1px solid #21262d;">\n          <div style="color: #8b949e; font-size: 10px; text-transform: uppercase;">LoRAs Active</div>\n          <div style="color: {\'#3fb950\' if loras_label != \'None\' else \'#8b949e\'}; font-weight: 600; margin-top: 2px;">{loras_label}</div>\n        </div>\n        <div style="background: #161b22; padding: 8px 12px; border-radius: 6px; border: 1px solid #21262d;">\n          <div style="color: #8b949e; font-size: 10px; text-transform: uppercase;">Elapsed Time</div>\n          <div style="color: #58a6ff; font-weight: 600; font-family: monospace; margin-top: 2px;">{elapsed_sec:.1f}s</div>\n        </div>\n      </div>\n\n      <div style="margin-bottom: 6px; display: flex; justify-content: space-between; font-size: 11px;">\n        <span style="color: #8b949e;">Pipeline Stage: <strong style="color: #f0f6fc;">{stage}</strong></span>\n        <span style="color: #58a6ff; font-family: monospace;">{bar_fill}%</span>\n      </div>\n      <div style="height: 6px; background: #21262d; border-radius: 3px; overflow: hidden;">\n        <div style="height: 100%; width: {bar_fill}%; background: {grad}; transition: width 0.15s ease-out;"></div>\n      </div>\n\n      <div style="margin-top: 10px; font-size: 11px; color: #8b949e; font-family: monospace;">\n        ⚡ Unit: <span style="color: #79c0ff;">{node_title}</span>\n      </div>\n    </div>\n    """\n    display_handle.update(HTML(html))\n\ndef display_comparison_preview(title: str, images_dict: dict):\n    cards_html = ""\n    for label, img in images_dict.items():\n        thumb = img.copy()\n        thumb.thumbnail((260, 260))\n        if thumb.mode in ("RGBA", "P"):\n            thumb = thumb.convert("RGB")\n        buf = io.BytesIO()\n        thumb.save(buf, format="JPEG", quality=85)\n        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")\n        cards_html += f"""\n        <div style="background: #161b22; border: 1px solid #30363d; border-radius: 8px; padding: 10px; text-align: center; flex: 1; max-width: 260px;">\n          <div style="font-size: 11px; font-weight: 600; color: #58a6ff; margin-bottom: 6px; text-transform: uppercase;">{label}</div>\n          <img src="data:image/jpeg;base64,{b64}" style="max-width: 100%; height: auto; border-radius: 6px; border: 1px solid #21262d; box-shadow: 0 4px 12px rgba(0,0,0,0.4);" />\n          <div style="font-size: 10px; color: #8b949e; margin-top: 6px; font-family: monospace;">{img.width} × {img.height} ({img.width/img.height:.2f}:1)</div>\n        </div>\n        """\n    html = f"""\n    <div style="font-family: -apple-system, BlinkMacSystemFont, \'Segoe UI\', Roboto, sans-serif; background: #0d1117; border: 1px solid #30363d; border-radius: 10px; padding: 14px; margin: 10px 0; max-width: 820px;">\n      <div style="font-size: 12px; font-weight: 600; color: #c9d1d9; margin-bottom: 10px;">📸 {title}</div>\n      <div style="display: flex; gap: 14px; flex-wrap: wrap;">\n        {cards_html}\n      </div>\n    </div>\n    """\n    display(HTML(html))\n\ndef load_raw_image(source_mode: str, path_or_url: str, default_filename: str) -> Image.Image:\n    input_dir = "/content/engine/input" if os.path.exists("/content/engine") else "input"\n    os.makedirs(input_dir, exist_ok=True)\n    dest_path = os.path.join(input_dir, default_filename)\n\n    img = None\n    if source_mode == "upload":\n        try:\n            from google.colab import files\n            print(f"Please select and upload image for \'{default_filename}\':")\n            uploaded = files.upload()\n            if uploaded:\n                first_key = list(uploaded.keys())[0]\n                img = Image.open(io.BytesIO(uploaded[first_key]))\n                with open(dest_path, "wb") as f:\n                    f.write(uploaded[first_key])\n        except Exception as e:\n            print(f"Colab upload prompt notice: {e}")\n\n        if img is None and os.path.exists(dest_path):\n            print(f"Notice: Reusing existing image at {dest_path} since no new file was uploaded.")\n            img = Image.open(dest_path)\n\n    elif source_mode == "path_or_url":\n        url_or_p = path_or_url.strip().strip(""\'")\n        if url_or_p:\n            if url_or_p.startswith("http://") or url_or_p.startswith("https://"):\n                req = urllib.request.Request(url_or_p, headers={"User-Agent": "Mozilla/5.0"})\n                raw_bytes = urllib.request.urlopen(req, timeout=30).read()\n                img = Image.open(io.BytesIO(raw_bytes))\n            elif os.path.exists(url_or_p):\n                img = Image.open(url_or_p)\n            else:\n                raise FileNotFoundError(f"Specified image path does not exist: {url_or_p}")\n        elif os.path.exists(dest_path):\n            print(f"Notice: Reusing existing image at {dest_path}.")\n            img = Image.open(dest_path)\n        else:\n            raise ValueError(f"Please enter an image path or URL for \'{default_filename}\'.")\n\n    if img is None:\n        raise ValueError(f"No image acquired for \'{default_filename}\'. Please upload a file or enter a valid URL/path.")\n\n    img = ImageOps.exif_transpose(img)\n    img = img.convert("RGB")\n    return img\n\ndef save_processed_image(img: Image.Image, target_filename: str, target_dims: tuple = None, target_area_mp: float = None):\n    input_dir = "/content/engine/input" if os.path.exists("/content/engine") else "input"\n    os.makedirs(input_dir, exist_ok=True)\n    name_root, ext = os.path.splitext(target_filename)\n    unique_name = f"{name_root}_{uuid.uuid4().hex[:6]}.png"\n    dest_path = os.path.join(input_dir, unique_name)\n\n    orig_w, orig_h = img.size\n    if target_dims is not None:\n        new_w, new_h = target_dims\n        new_w = max(64, int(round(new_w / 32) * 32))\n        new_h = max(64, int(round(new_h / 32) * 32))\n        proc_img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)\n    elif target_area_mp is not None and target_area_mp > 0:\n        total_px = target_area_mp * 1024 * 1024\n        ratio = orig_w / orig_h\n        scale = math.sqrt(total_px / (ratio if ratio > 0 else 1.0))\n        new_w = max(64, int(round(ratio * scale / 32) * 32))\n        new_h = max(64, int(round(scale / 32) * 32))\n        proc_img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)\n    else:\n        new_w = max(64, int(round(orig_w / 32) * 32))\n        new_h = max(64, int(round(orig_h / 32) * 32))\n        proc_img = img.resize((new_w, new_h), Image.Resampling.LANCZOS) if (new_w, new_h) != (orig_w, orig_h) else img\n\n    proc_img.save(dest_path, "PNG")\n    try:\n        proc_img.save(os.path.join(input_dir, target_filename), "PNG")\n    except Exception:\n        pass\n    return unique_name, proc_img.width, proc_img.height\n\ndef calculate_target_dimensions(dimension_mode: str, orig_w: int, orig_h: int, megapixels: float, preset_ratio_str: str = "1:1 (Square)", custom_w: int = 1024, custom_h: int = 1024):\n    if dimension_mode.startswith("Match"):\n        ratio = orig_w / orig_h\n        total_pixels = megapixels * 1024 * 1024\n        scale = math.sqrt(total_pixels / (ratio if ratio > 0 else 1.0))\n        target_w = int(round(ratio * scale / 32) * 32)\n        target_h = int(round(scale / 32) * 32)\n    elif dimension_mode == "Custom Dimensions":\n        target_w = int(round(custom_w / 32) * 32)\n        target_h = int(round(custom_h / 32) * 32)\n    else:\n        ASPECT_RATIO_SPECS = {\n            "1:1 (Square)": (1, 1),\n            "16:9 (Widescreen)": (16, 9),\n            "9:16 (Portrait Widescreen)": (9, 16),\n            "21:9 (Ultrawide)": (21, 9),\n            "9:21 (Tall Ultrawide)": (9, 21),\n            "4:3 (Standard)": (4, 3),\n            "3:4 (Portrait Standard)": (3, 4),\n            "3:2 (Photo)": (3, 2),\n            "2:3 (Portrait Photo)": (2, 3),\n            "5:4 (Landscape Tall)": (5, 4),\n            "4:5 (Portrait Tall)": (4, 5),\n            "1:2 (Tall Vertical)": (1, 2),\n            "2:1 (Panoramic)": (2, 1)\n        }\n        w_ratio, h_ratio = ASPECT_RATIO_SPECS.get(preset_ratio_str, (1, 1))\n        total_pixels = megapixels * 1024 * 1024\n        scale = math.sqrt(total_pixels / (w_ratio * h_ratio))\n        target_w = int(round(w_ratio * scale / 32) * 32)\n        target_h = int(round(h_ratio * scale / 32) * 32)\n\n    target_w = max(64, min(2048, target_w))\n    target_h = max(64, min(2048, target_h))\n    return target_w, target_h\n\ndef resolve_lora_selection(selected_val: str, custom_val: str = "") -> str:\n    if not selected_val or selected_val == "None":\n        return None\n    if selected_val.startswith("Custom"):\n        c_name = custom_val.strip().strip(""\'")\n        if not c_name:\n            return None\n        c_name = os.path.basename(c_name)\n        if not (c_name.endswith(".safetensors") or c_name.endswith(".pt") or c_name.endswith(".bin")):\n            c_name += ".safetensors"\n        return c_name\n    return selected_val.strip()\n\ndef download_lora_multisource(url: str, output_dir: str, exact_filename: str, civ_token: str = "", hf_tok: str = ""):\n    os.makedirs(output_dir, exist_ok=True)\n    target_file = os.path.join(output_dir, exact_filename)\n    aria_meta_file = target_file + ".aria2"\n\n    if os.path.exists(target_file) and not os.path.exists(aria_meta_file):\n        size_mb = os.path.getsize(target_file) / (1024 * 1024)\n        if size_mb >= 1.0:\n            return "Existing / Verified", size_mb\n\n    clean_url = url.strip()\n    platform = "Direct Link"\n\n    if "drive.google.com" in clean_url or "docs.google.com" in clean_url:\n        platform = "Google Drive"\n        import gdown\n        gdown.download(clean_url, target_file, fuzzy=True, quiet=False)\n\n    elif "civitai.com" in clean_url:\n        platform = "Civitai"\n        m_ver = re.search(r"modelVersionId=([0-9]+)", clean_url)\n        if m_ver:\n            clean_url = f"https://civitai.com/api/download/models/{m_ver.group(1)}"\n        elif "/models/" in clean_url and "/api/" not in clean_url:\n            m_id = re.search(r"/models/([0-9]+)", clean_url)\n            if m_id:\n                try:\n                    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}\n                    if civ_token.strip():\n                        headers["Authorization"] = f"Bearer {civ_token.strip()}"\n                    res = requests.get(f"https://civitai.com/api/v1/models/{m_id.group(1)}", headers=headers, timeout=6).json()\n                    ver_id = res["modelVersions"][0]["id"]\n                    clean_url = f"https://civitai.com/api/download/models/{ver_id}"\n                except Exception as e:\n                    print(f"Notice: Civitai auto-resolution defaulted to URL: {e}")\n\n        header_flags = ["--header=User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64)"]\n        if civ_token.strip():\n            header_flags.append(f"--header=Authorization: Bearer {civ_token.strip()}")\n            delimiter = "&" if "?" in clean_url else "?"\n            clean_url = f"{clean_url}{delimiter}token={civ_token.strip()}"\n\n        cmd = [\n            "aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M",\n            "--auto-file-renaming=false", "--allow-overwrite=true", "--content-disposition=false",\n            "-d", output_dir, "-o", exact_filename, *header_flags, clean_url\n        ]\n        subprocess.run(cmd)\n\n    elif "huggingface.co" in clean_url:\n        platform = "Hugging Face"\n        clean_url = clean_url.replace("/blob/", "/resolve/")\n        header_flags = []\n        if hf_tok.strip():\n            header_flags.append(f"--header=Authorization: Bearer {hf_tok.strip()}")\n\n        cmd = [\n            "aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M",\n            "--auto-file-renaming=false", "--allow-overwrite=true", "--content-disposition=false",\n            "-d", output_dir, "-o", exact_filename, *header_flags, clean_url\n        ]\n        subprocess.run(cmd)\n\n    else:\n        cmd = [\n            "aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M",\n            "--auto-file-renaming=false", "--allow-overwrite=true", "--content-disposition=false",\n            "-d", output_dir, "-o", exact_filename, clean_url\n        ]\n        subprocess.run(cmd)\n\n    if not os.path.exists(target_file) or os.path.getsize(target_file) < 1024 * 1024:\n        raise RuntimeError(f"Failed to download LoRA from {platform}. Please check URL or file permissions.")\n\n    if os.path.exists(aria_meta_file):\n        os.remove(aria_meta_file)\n\n    size_mb = os.path.getsize(target_file) / (1024 * 1024)\n    return platform, size_mb\n\ndef build_lora_chain(workflow, active_loras, base_model_link, base_clip_link):\n    current_model = base_model_link\n    current_clip = base_clip_link\n    loras_dir = "/content/engine/models/loras" if os.path.exists("/content/engine") else "models/loras"\n    os.makedirs(loras_dir, exist_ok=True)\n\n    PRESET_URLS = {\n        "bfs_head_v1.1_qwen_2.1.safetensors": "https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/resolve/main/bfs_head_v1.1_qwen_2.1.safetensors",\n        "bfs_body_swap_v1.0_qwen_2.1.safetensors": "https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/resolve/main/bfs_body_swap_v1.0_qwen_2.1.safetensors",\n        "Qwen2.1_Anything2RealCharacters.safetensors": "https://huggingface.co/WarmBloodAban/Qwen-Image-2.1-LoRAs/resolve/main/Qwen2.1_Anything2RealCharacters.safetensors",\n        "p_qwen_image_2.1_8step_v0.1.safetensors": "https://huggingface.co/PrunaAI/Pruna-Qwen-Image-2.1/resolve/main/p_qwen_image_2.1_8step_v0.1.safetensors"\n    }\n\n    for idx, lora in enumerate(active_loras):\n        node_id = f"lora_loader_{idx+1}"\n        clean_name = os.path.basename(lora["name"].strip())\n        local_path = os.path.join(loras_dir, clean_name)\n\n        if not os.path.exists(local_path) and clean_name in PRESET_URLS:\n            print(f"📥 Auto-downloading missing LoRA \'{clean_name}\'...")\n            download_lora_multisource(PRESET_URLS[clean_name], loras_dir, clean_name)\n\n        if not os.path.exists(local_path):\n            print(f"⚠️ Warning: LoRA \'{clean_name}\' was not detected in {loras_dir}. Make sure you ran Cell 2 to download it.")\n\n        workflow[node_id] = {\n            "inputs": {\n                "lora_name": clean_name,\n                "strength_model": float(lora.get("strength_model", 1.0)),\n                "strength_clip": float(lora.get("strength_clip", 1.0)),\n                "model": current_model,\n                "clip": current_clip\n            },\n            "class_type": "LoraLoader",\n            "_meta": {"title": f"LoRA Loader #{idx+1} ({clean_name})"}\n        }\n        current_model = [node_id, 0]\n        current_clip = [node_id, 1]\n\n    return current_model, current_clip\n\ndef execute_comfy_generation(workflow, task_name, target_width, target_height, seed, active_loras_summary):\n    ensure_server_alive()\n    purge_backend_memory()\n\n    dims_label = f"{target_width} × {target_height}"\n    display_handle = display(HTML(""), display_id=str(uuid.uuid4()))\n    render_dashboard(display_handle, task_name, "Validating & Submitting Prompt...", "Dispatcher", 0, 0.0, seed, dims_label, active_loras_summary)\n\n    start_t = time.time()\n    prompt_res = queue_prompt(workflow)\n    prompt_id = prompt_res["prompt_id"]\n\n    ws = websocket.WebSocket()\n    ws.connect(f"ws://{server_address}/ws?clientId={client_id}")\n    ws.settimeout(1.0)\n    current_pct = 5\n\n    try:\n        while True:\n            if "server_proc" in globals() and server_proc is not None and server_proc.poll() is not None:\n                render_dashboard(display_handle, task_name, "Server Crashed", "Terminated", current_pct, time.time() - start_t, seed, dims_label, active_loras_summary, error=True)\n                raise RuntimeError("Backend server process terminated unexpectedly.")\n\n            try:\n                out = ws.recv()\n            except websocket.WebSocketTimeoutException:\n                render_dashboard(display_handle, task_name, f"Computing Diffusion Steps ({dims_label})...", "In Progress", current_pct, time.time() - start_t, seed, dims_label, active_loras_summary)\n                continue\n\n            if isinstance(out, str):\n                msg = json.loads(out)\n                msg_type = msg.get("type")\n                elapsed = time.time() - start_t\n\n                if msg_type == "execution_error":\n                    err_info = msg.get("data", {})\n                    err_msg = err_info.get("exception_message", "Execution error")\n                    render_dashboard(display_handle, task_name, f"Error in {err_info.get(\'node_type\')}", err_msg, current_pct, elapsed, seed, dims_label, active_loras_summary, error=True)\n                    raise RuntimeError(f"Pipeline Execution Error: {err_msg}")\n\n                if msg_type == "execution_interrupted":\n                    render_dashboard(display_handle, task_name, "Canceled", "Interrupted", current_pct, elapsed, seed, dims_label, active_loras_summary, error=True)\n                    break\n\n                if msg_type == "executing":\n                    node_id = msg["data"].get("node")\n                    msg_prompt_id = msg["data"].get("prompt_id")\n                    if node_id is None:\n                        if msg_prompt_id is None or msg_prompt_id == prompt_id:\n                            render_dashboard(display_handle, task_name, f"Completed ({dims_label})", "Latent Decoded", 100, elapsed, seed, dims_label, active_loras_summary, done=True)\n                            break\n                    elif str(node_id) in workflow:\n                        node_name = workflow[str(node_id)].get("_meta", {}).get("title", f"Node {node_id}")\n                        render_dashboard(display_handle, task_name, f"Processing ({dims_label})", node_name, current_pct, elapsed, seed, dims_label, active_loras_summary)\n\n                elif msg_type == "progress":\n                    data = msg["data"]\n                    step = data["value"]\n                    total_steps = data["max"]\n                    current_pct = 10 + int((step / total_steps) * 85)\n                    render_dashboard(display_handle, task_name, f"Sampling {step}/{total_steps} ({dims_label})", "KSampler (Euler Simple)", current_pct, elapsed, seed, dims_label, active_loras_summary)\n    finally:\n        try:\n            ws.close()\n        except Exception:\n            pass\n\n    history = None\n    for _ in range(10):\n        try:\n            history_req = urllib.request.urlopen(f"http://{server_address}/history/{prompt_id}")\n            history_data = json.loads(history_req.read().decode("utf-8"))\n            if prompt_id in history_data:\n                history = history_data[prompt_id]\n                break\n        except Exception:\n            pass\n        time.sleep(0.5)\n\n    if not history:\n        raise RuntimeError(f"Failed to fetch execution history for prompt {prompt_id} from backend server.")\n\n    output_images = []\n    for node_id in history.get("outputs", {}):\n        node_output = history["outputs"][node_id]\n        if "images" in node_output:\n            for img_info in node_output["images"]:\n                raw_data = get_image(img_info["filename"], img_info["subfolder"], img_info["type"])\n                output_images.append(Image.open(io.BytesIO(raw_data)))\n\n    purge_backend_memory()\n\n    save_dir = "/content/output"\n    os.makedirs(save_dir, exist_ok=True)\n\n    for i, img in enumerate(output_images):\n        clean_task = task_name.lower().replace(" ", "_").replace("-", "_")\n        out_filename = f"{clean_task}_{seed}_{i+1}.png"\n        out_full_path = os.path.join(save_dir, out_filename)\n        img.save(out_full_path, "PNG")\n\n        display(HTML(f"""\n        <div style="font-family: -apple-system, BlinkMacSystemFont, \'Segoe UI\', Roboto, sans-serif; background: #0d1117; border: 1px solid #30363d; border-radius: 8px; padding: 12px; margin-top: 10px; max-width: 820px;">\n          <div style="display: flex; justify-content: space-between; align-items: center; color: #8b949e; font-size: 11px;">\n            <span>Resolution: <strong style="color: #f0f6fc;">{img.width} × {img.height}</strong> | Render: <strong style="color: #58a6ff;">{time.time() - start_t:.2f}s</strong> | Seed: <strong style="color: #f0f6fc;">{seed}</strong></span>\n            <span>💾 Saved to: <strong style="color: #79c0ff; font-family: monospace;">{out_full_path}</strong></span>\n          </div>\n        </div>\n        """))\n        display(img)\n    return output_images\n'

for p in ["/content/engine/qwen_studio_utils.py", "/content/qwen_studio_utils.py"]:
    try:
        with open(p, "w", encoding="utf-8") as f:
            f.write(SHARED_UTILS_CODE)
    except Exception:
        pass

for p in ["/content/engine", "/content"]:
    if p not in sys.path:
        sys.path.insert(0, p)

table_rows = ""
for item in models_manifest:
    status, size = foolproof_download(item["url"], item["dir"], item["file"], item["min_mb"])
    badge_color = "#3fb950" if status == "verified" else "#58a6ff"
    badge_text = "VERIFIED" if status == "verified" else "SYNCED"
    table_rows += f"""
    <tr style="border-bottom: 1px solid #21262d;">
      <td style="padding: 8px 12px; color: #c9d1d9;">{item['name']}</td>
      <td style="padding: 8px 12px; color: #8b949e; font-family: monospace;">{item['file']}</td>
      <td style="padding: 8px 12px; color: #8b949e; text-align: right;">{size:.1f} MB</td>
      <td style="padding: 8px 12px; text-align: center;">
        <span style="background: rgba(46, 160, 67, 0.15); color: {badge_color}; border: 1px solid {badge_color}40; padding: 2px 8px; border-radius: 12px; font-size: 10px; font-weight: 600;">{badge_text}</span>
      </td>
    </tr>
    """

total_elapsed = time.time() - t0
display(HTML(f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background: #0d1117; border: 1px solid #30363d; border-radius: 10px; padding: 18px; max-width: 820px; box-shadow: 0 8px 24px rgba(0,0,0,0.3); margin: 10px 0;">
  <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #21262d; padding-bottom: 12px; margin-bottom: 12px;">
    <div style="display: flex; align-items: center; gap: 8px;">
      <span style="width: 10px; height: 10px; border-radius: 50%; background: #3fb950; box-shadow: 0 0 10px #238636;"></span>
      <span style="color: #f0f6fc; font-weight: 600; font-size: 13px;">Qwen-Image-2.1 Base Weights & Registry</span>
    </div>
    <span style="background: rgba(56, 139, 253, 0.15); color: #58a6ff; border: 1px solid rgba(56, 139, 253, 0.4); padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: 600;">
      READY ({total_elapsed:.1f}s)
    </span>
  </div>

  <table style="width: 100%; border-collapse: collapse; font-size: 12px;">
    <thead>
      <tr style="border-bottom: 1px solid #30363d; color: #8b949e; text-align: left;">
        <th style="padding: 6px 12px;">Module</th>
        <th style="padding: 6px 12px;">Target Filename</th>
        <th style="padding: 6px 12px; text-align: right;">Size</th>
        <th style="padding: 6px 12px; text-align: center;">Status</th>
      </tr>
    </thead>
    <tbody>
      {table_rows}
    </tbody>
  </table>
</div>
"""))


In [ ]:
# @title 2. Preset LoRAs & Custom LoRA Downloader
# @markdown Select which default high-performance LoRAs you want to acquire:
download_bfs_head = True # @param {type:"boolean"}
download_bfs_body = True # @param {type:"boolean"}
download_anything2real = True # @param {type:"boolean"}
download_turbo_8step = True # @param {type:"boolean"}

# @markdown ---
# @markdown #### 🔗 Optional: Download Custom LoRA from Civitai / Hugging Face / Google Drive
custom_lora_url = "" # @param {type:"string"}
custom_lora_filename = "custom_lora.safetensors" # @param {type:"string"}
civitai_token = "" # @param {type:"string"}
hf_token = "" # @param {type:"string"}

import os
import sys
from IPython.display import display, HTML

for _p in ["/content/engine", "/content", "."]:
    if _p not in sys.path and os.path.exists(_p):
        sys.path.insert(0, _p)

try:
    from qwen_studio_utils import download_lora_multisource
except ImportError:
    pass

loras_dir = "/content/engine/models/loras"
os.makedirs(loras_dir, exist_ok=True)

# Preset LoRA Definitions
preset_loras = [
    {
        "id": "bfs_head",
        "enabled": download_bfs_head,
        "name": "BFS Head Swap V1.1 (Face & Head Replacement)",
        "file": "bfs_head_v1.1_qwen_2.1.safetensors",
        "url": "https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/resolve/main/bfs_head_v1.1_qwen_2.1.safetensors",
        "min_mb": 50.0,
        "desc": "Recommended for Head Swap (Cell 6). Transfers facial identity while preserving pose, expression and lighting."
    },
    {
        "id": "bfs_body",
        "enabled": download_bfs_body,
        "name": "BFS Body Swap V1.0 (Full Body & Outfit Transfer)",
        "file": "bfs_body_swap_v1.0_qwen_2.1.safetensors",
        "url": "https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/resolve/main/bfs_body_swap_v1.0_qwen_2.1.safetensors",
        "min_mb": 50.0,
        "desc": "Recommended for Body Swap (Cell 7). Replaces body and clothing while preserving scene background and pose."
    },
    {
        "id": "anything2real",
        "enabled": download_anything2real,
        "name": "Anything to Real Characters (Style to Photorealism)",
        "file": "Qwen2.1_Anything2RealCharacters.safetensors",
        "url": "https://huggingface.co/WarmBloodAban/Qwen-Image-2.1-LoRAs/resolve/main/Qwen2.1_Anything2RealCharacters.safetensors",
        "min_mb": 50.0,
        "desc": "Recommended for Image Edit (Cell 5). Turns anime, cartoon, sketch, or 3D characters into realistic humans."
    },
    {
        "id": "turbo_8step",
        "enabled": download_turbo_8step,
        "name": "Pruna 8-Step Turbo Acceleration LoRA",
        "file": "p_qwen_image_2.1_8step_v0.1.safetensors",
        "url": "https://huggingface.co/PrunaAI/Pruna-Qwen-Image-2.1/resolve/main/p_qwen_image_2.1_8step_v0.1.safetensors",
        "min_mb": 100.0,
        "desc": "Distilled turbo adapter. Drastically accelerates sampling to 8 steps for T2I, Image Edit, Head and Body Swap!"
    }
]

# Download selected preset LoRAs
results_rows = ""
for item in preset_loras:
    if item["enabled"]:
        status, size_mb = download_lora_multisource(item["url"], loras_dir, item["file"])
        results_rows += f"""
        <tr style="border-bottom: 1px solid #21262d;">
          <td style="padding: 8px 12px; color: #f0f6fc; font-weight: 500;">{item['name']}</td>
          <td style="padding: 8px 12px; color: #79c0ff; font-family: monospace;">{item['file']}</td>
          <td style="padding: 8px 12px; color: #8b949e; text-align: right;">{size_mb:.1f} MB</td>
          <td style="padding: 8px 12px; text-align: center;">
            <span style="background: rgba(46, 160, 67, 0.15); color: #3fb950; border: 1px solid #3fb95040; padding: 2px 8px; border-radius: 12px; font-size: 10px; font-weight: 600;">ACTIVE</span>
          </td>
        </tr>
        """

# Download optional custom LoRA
if custom_lora_url.strip():
    plat, csize = download_lora_multisource(custom_lora_url.strip(), loras_dir, custom_lora_filename.strip(), civitai_token, hf_token)
    results_rows += f"""
    <tr style="border-bottom: 1px solid #21262d;">
      <td style="padding: 8px 12px; color: #d2a8ff; font-weight: 500;">Custom LoRA ({plat})</td>
      <td style="padding: 8px 12px; color: #d2a8ff; font-family: monospace;">{custom_lora_filename.strip()}</td>
      <td style="padding: 8px 12px; color: #8b949e; text-align: right;">{csize:.1f} MB</td>
      <td style="padding: 8px 12px; text-align: center;">
        <span style="background: rgba(163, 113, 247, 0.15); color: #d2a8ff; border: 1px solid #a371f740; padding: 2px 8px; border-radius: 12px; font-size: 10px; font-weight: 600;">CUSTOM</span>
      </td>
    </tr>
    """

# Check all installed LoRAs
all_files = sorted(os.listdir(loras_dir)) if os.path.exists(loras_dir) else []
total_loras = len(all_files)

display(HTML(f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background: #0d1117; border: 1px solid #30363d; border-radius: 10px; padding: 18px; max-width: 820px; box-shadow: 0 8px 24px rgba(0,0,0,0.3); margin: 10px 0;">
  <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #21262d; padding-bottom: 12px; margin-bottom: 12px;">
    <div style="display: flex; align-items: center; gap: 8px;">
      <span style="width: 10px; height: 10px; border-radius: 50%; background: #a371f7; box-shadow: 0 0 10px #8957e5;"></span>
      <span style="color: #f0f6fc; font-weight: 600; font-size: 13px;">LoRA Registry & Multi-LoRA Storage</span>
    </div>
    <span style="background: rgba(163, 113, 247, 0.15); color: #d2a8ff; border: 1px solid rgba(163, 113, 247, 0.4); padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: 600;">
      {total_loras} LORAS AVAILABLE
    </span>
  </div>

  <table style="width: 100%; border-collapse: collapse; font-size: 12px;">
    <thead>
      <tr style="border-bottom: 1px solid #30363d; color: #8b949e; text-align: left;">
        <th style="padding: 6px 12px;">LoRA Preset / Name</th>
        <th style="padding: 6px 12px;">Filename</th>
        <th style="padding: 6px 12px; text-align: right;">Size</th>
        <th style="padding: 6px 12px; text-align: center;">Status</th>
      </tr>
    </thead>
    <tbody>
      {results_rows}
    </tbody>
  </table>
  <div style="margin-top: 10px; font-size: 11px; color: #8b949e; font-family: monospace;">
    📁 Stored at: <span style="color: #58a6ff;">/content/engine/models/loras/</span>
  </div>
</div>
"""))


In [ ]:
# @title 3. Start Inference Server & Common Engine Core
import os
import sys
import time
from IPython.display import display, HTML

# 1. Module Discovery & Self-Healing
for _p in ["/content/engine", "/content", "."]:
    if _p not in sys.path and os.path.exists(_p):
        sys.path.insert(0, _p)

SHARED_UTILS_CODE = 'import os\nimport sys\nimport time\nimport json\nimport math\nimport uuid\nimport random\nimport io\nimport re\nimport subprocess\nimport requests\nimport urllib.request\nimport urllib.parse\nimport websocket\nimport base64\nfrom PIL import Image, ImageOps\nfrom IPython.display import display, HTML\n\n# Host memory conservation flags\nos.environ["TORCH_ALLOW_TF32_CUBLAS_OVERRIDE"] = "1"\nos.environ["CUDA_MODULE_LOADING"] = "LAZY"\nos.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"\n\nserver_address = "127.0.0.1:8188"\nclient_id = str(uuid.uuid4())\nserver_proc = None\n\ndef launch_inference_server():\n    global server_proc\n    engine_dir = "/content/engine" if os.path.exists("/content/engine") else "."\n    server_proc = subprocess.Popen(\n        [\n            "python", "main.py",\n            "--listen", "127.0.0.1",\n            "--port", "8188",\n            "--disable-pinned-memory",\n            "--mmap-torch-files",\n            "--fast",\n            "--cuda-malloc",\n            "--fp16-vae",\n            "--preview-method", "none"\n        ],\n        cwd=engine_dir,\n        stdout=subprocess.PIPE,\n        stderr=subprocess.STDOUT,\n        text=True,\n        bufsize=1\n    )\n    return server_proc\n\ndef ensure_server_alive():\n    global server_address\n    try:\n        r = requests.get(f"http://{server_address}/system_stats", timeout=1.5)\n        if r.status_code == 200:\n            return True\n    except requests.exceptions.RequestException:\n        pass\n\n    print("Backend server is not running. Launching inference server on 127.0.0.1:8188...")\n    launch_inference_server()\n    t_wait = time.time()\n    while time.time() - t_wait < 60:\n        try:\n            r = requests.get(f"http://{server_address}/system_stats", timeout=1.5)\n            if r.status_code == 200:\n                print(f"✓ Backend server initialized and online ({time.time() - t_wait:.1f}s).")\n                return True\n        except requests.exceptions.RequestException:\n            pass\n        time.sleep(1.2)\n    raise RuntimeError("Failed to connect to backend server on 127.0.0.1:8188 after 60s.")\n\ndef purge_backend_memory():\n    try:\n        payload = json.dumps({"unload_models": False, "free_memory": True}).encode("utf-8")\n        req = urllib.request.Request(f"http://{server_address}/free", data=payload, headers={"Content-Type": "application/json"})\n        with urllib.request.urlopen(req, timeout=5) as resp:\n            pass\n    except Exception:\n        pass\n\ndef queue_prompt(prompt_workflow):\n    payload = json.dumps({"prompt": prompt_workflow, "client_id": client_id}).encode("utf-8")\n    req = urllib.request.Request(f"http://{server_address}/prompt", data=payload, headers={"Content-Type": "application/json"})\n    try:\n        with urllib.request.urlopen(req) as resp:\n            res_data = json.loads(resp.read().decode("utf-8"))\n            if "error" in res_data:\n                raise ValueError(f"Prompt validation error: {res_data[\'error\']}")\n            if "node_errors" in res_data and res_data["node_errors"]:\n                raise ValueError(f"Node error in workflow: {res_data[\'node_errors\']}")\n            if "prompt_id" not in res_data:\n                raise ValueError(f"Server rejected prompt without ID: {res_data}")\n            return res_data\n    except urllib.error.HTTPError as e:\n        err_body = e.read().decode("utf-8")\n        raise ValueError(f"Server rejected prompt (HTTP {e.code}): {err_body}")\n\ndef get_image(filename, subfolder, folder_type):\n    params = urllib.parse.urlencode({"filename": filename, "subfolder": subfolder, "type": folder_type})\n    with urllib.request.urlopen(f"http://{server_address}/view?{params}") as response:\n        return response.read()\n\ndef render_dashboard(display_handle, task_name: str, stage: str, node_title: str, progress_pct: int, elapsed_sec: float, seed: int, dims: str, loras_label: str, done: bool = False, error: bool = False):\n    if error:\n        badge_color = "#f85149"\n        status_text = "ERROR"\n        grad = "#da3633"\n    elif done:\n        badge_color = "#3fb950"\n        status_text = "COMPLETED"\n        grad = "#2ea043"\n    else:\n        badge_color = "#58a6ff"\n        status_text = "PROCESSING"\n        grad = "linear-gradient(90deg, #1f6feb, #58a6ff)"\n\n    bar_fill = min(100, max(0, progress_pct))\n\n    html = f"""\n    <div style="font-family: -apple-system, BlinkMacSystemFont, \'Segoe UI\', Roboto, sans-serif; background: #0d1117; color: #e6edf3; border: 1px solid #30363d; border-radius: 10px; padding: 18px; margin: 12px 0; max-width: 820px; box-shadow: 0 8px 24px rgba(0,0,0,0.3);">\n      <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 14px; border-bottom: 1px solid #21262d; padding-bottom: 10px;">\n        <span style="font-size: 13px; font-weight: 600; letter-spacing: 0.5px; color: #58a6ff;">✦ {task_name.upper()}</span>\n        <span style="background: rgba(88, 166, 255, 0.12); color: {badge_color}; border: 1px solid {badge_color}40; padding: 3px 10px; border-radius: 20px; font-size: 11px; font-weight: 600;">{status_text}</span>\n      </div>\n\n      <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; font-size: 12px; margin-bottom: 14px;">\n        <div style="background: #161b22; padding: 8px 12px; border-radius: 6px; border: 1px solid #21262d;">\n          <div style="color: #8b949e; font-size: 10px; text-transform: uppercase;">Seed</div>\n          <div style="color: #f0f6fc; font-weight: 600; font-family: monospace; margin-top: 2px;">{seed}</div>\n        </div>\n        <div style="background: #161b22; padding: 8px 12px; border-radius: 6px; border: 1px solid #21262d;">\n          <div style="color: #8b949e; font-size: 10px; text-transform: uppercase;">Canvas / Dims</div>\n          <div style="color: #f0f6fc; font-weight: 600; margin-top: 2px;">{dims}</div>\n        </div>\n        <div style="background: #161b22; padding: 8px 12px; border-radius: 6px; border: 1px solid #21262d;">\n          <div style="color: #8b949e; font-size: 10px; text-transform: uppercase;">LoRAs Active</div>\n          <div style="color: {\'#3fb950\' if loras_label != \'None\' else \'#8b949e\'}; font-weight: 600; margin-top: 2px;">{loras_label}</div>\n        </div>\n        <div style="background: #161b22; padding: 8px 12px; border-radius: 6px; border: 1px solid #21262d;">\n          <div style="color: #8b949e; font-size: 10px; text-transform: uppercase;">Elapsed Time</div>\n          <div style="color: #58a6ff; font-weight: 600; font-family: monospace; margin-top: 2px;">{elapsed_sec:.1f}s</div>\n        </div>\n      </div>\n\n      <div style="margin-bottom: 6px; display: flex; justify-content: space-between; font-size: 11px;">\n        <span style="color: #8b949e;">Pipeline Stage: <strong style="color: #f0f6fc;">{stage}</strong></span>\n        <span style="color: #58a6ff; font-family: monospace;">{bar_fill}%</span>\n      </div>\n      <div style="height: 6px; background: #21262d; border-radius: 3px; overflow: hidden;">\n        <div style="height: 100%; width: {bar_fill}%; background: {grad}; transition: width 0.15s ease-out;"></div>\n      </div>\n\n      <div style="margin-top: 10px; font-size: 11px; color: #8b949e; font-family: monospace;">\n        ⚡ Unit: <span style="color: #79c0ff;">{node_title}</span>\n      </div>\n    </div>\n    """\n    display_handle.update(HTML(html))\n\ndef display_comparison_preview(title: str, images_dict: dict):\n    cards_html = ""\n    for label, img in images_dict.items():\n        thumb = img.copy()\n        thumb.thumbnail((260, 260))\n        if thumb.mode in ("RGBA", "P"):\n            thumb = thumb.convert("RGB")\n        buf = io.BytesIO()\n        thumb.save(buf, format="JPEG", quality=85)\n        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")\n        cards_html += f"""\n        <div style="background: #161b22; border: 1px solid #30363d; border-radius: 8px; padding: 10px; text-align: center; flex: 1; max-width: 260px;">\n          <div style="font-size: 11px; font-weight: 600; color: #58a6ff; margin-bottom: 6px; text-transform: uppercase;">{label}</div>\n          <img src="data:image/jpeg;base64,{b64}" style="max-width: 100%; height: auto; border-radius: 6px; border: 1px solid #21262d; box-shadow: 0 4px 12px rgba(0,0,0,0.4);" />\n          <div style="font-size: 10px; color: #8b949e; margin-top: 6px; font-family: monospace;">{img.width} × {img.height} ({img.width/img.height:.2f}:1)</div>\n        </div>\n        """\n    html = f"""\n    <div style="font-family: -apple-system, BlinkMacSystemFont, \'Segoe UI\', Roboto, sans-serif; background: #0d1117; border: 1px solid #30363d; border-radius: 10px; padding: 14px; margin: 10px 0; max-width: 820px;">\n      <div style="font-size: 12px; font-weight: 600; color: #c9d1d9; margin-bottom: 10px;">📸 {title}</div>\n      <div style="display: flex; gap: 14px; flex-wrap: wrap;">\n        {cards_html}\n      </div>\n    </div>\n    """\n    display(HTML(html))\n\ndef load_raw_image(source_mode: str, path_or_url: str, default_filename: str) -> Image.Image:\n    input_dir = "/content/engine/input" if os.path.exists("/content/engine") else "input"\n    os.makedirs(input_dir, exist_ok=True)\n    dest_path = os.path.join(input_dir, default_filename)\n\n    img = None\n    if source_mode == "upload":\n        try:\n            from google.colab import files\n            print(f"Please select and upload image for \'{default_filename}\':")\n            uploaded = files.upload()\n            if uploaded:\n                first_key = list(uploaded.keys())[0]\n                img = Image.open(io.BytesIO(uploaded[first_key]))\n                with open(dest_path, "wb") as f:\n                    f.write(uploaded[first_key])\n        except Exception as e:\n            print(f"Colab upload prompt notice: {e}")\n\n        if img is None and os.path.exists(dest_path):\n            print(f"Notice: Reusing existing image at {dest_path} since no new file was uploaded.")\n            img = Image.open(dest_path)\n\n    elif source_mode == "path_or_url":\n        url_or_p = path_or_url.strip().strip(""\'")\n        if url_or_p:\n            if url_or_p.startswith("http://") or url_or_p.startswith("https://"):\n                req = urllib.request.Request(url_or_p, headers={"User-Agent": "Mozilla/5.0"})\n                raw_bytes = urllib.request.urlopen(req, timeout=30).read()\n                img = Image.open(io.BytesIO(raw_bytes))\n            elif os.path.exists(url_or_p):\n                img = Image.open(url_or_p)\n            else:\n                raise FileNotFoundError(f"Specified image path does not exist: {url_or_p}")\n        elif os.path.exists(dest_path):\n            print(f"Notice: Reusing existing image at {dest_path}.")\n            img = Image.open(dest_path)\n        else:\n            raise ValueError(f"Please enter an image path or URL for \'{default_filename}\'.")\n\n    if img is None:\n        raise ValueError(f"No image acquired for \'{default_filename}\'. Please upload a file or enter a valid URL/path.")\n\n    img = ImageOps.exif_transpose(img)\n    img = img.convert("RGB")\n    return img\n\ndef save_processed_image(img: Image.Image, target_filename: str, target_dims: tuple = None, target_area_mp: float = None):\n    input_dir = "/content/engine/input" if os.path.exists("/content/engine") else "input"\n    os.makedirs(input_dir, exist_ok=True)\n    name_root, ext = os.path.splitext(target_filename)\n    unique_name = f"{name_root}_{uuid.uuid4().hex[:6]}.png"\n    dest_path = os.path.join(input_dir, unique_name)\n\n    orig_w, orig_h = img.size\n    if target_dims is not None:\n        new_w, new_h = target_dims\n        new_w = max(64, int(round(new_w / 32) * 32))\n        new_h = max(64, int(round(new_h / 32) * 32))\n        proc_img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)\n    elif target_area_mp is not None and target_area_mp > 0:\n        total_px = target_area_mp * 1024 * 1024\n        ratio = orig_w / orig_h\n        scale = math.sqrt(total_px / (ratio if ratio > 0 else 1.0))\n        new_w = max(64, int(round(ratio * scale / 32) * 32))\n        new_h = max(64, int(round(scale / 32) * 32))\n        proc_img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)\n    else:\n        new_w = max(64, int(round(orig_w / 32) * 32))\n        new_h = max(64, int(round(orig_h / 32) * 32))\n        proc_img = img.resize((new_w, new_h), Image.Resampling.LANCZOS) if (new_w, new_h) != (orig_w, orig_h) else img\n\n    proc_img.save(dest_path, "PNG")\n    try:\n        proc_img.save(os.path.join(input_dir, target_filename), "PNG")\n    except Exception:\n        pass\n    return unique_name, proc_img.width, proc_img.height\n\ndef calculate_target_dimensions(dimension_mode: str, orig_w: int, orig_h: int, megapixels: float, preset_ratio_str: str = "1:1 (Square)", custom_w: int = 1024, custom_h: int = 1024):\n    if dimension_mode.startswith("Match"):\n        ratio = orig_w / orig_h\n        total_pixels = megapixels * 1024 * 1024\n        scale = math.sqrt(total_pixels / (ratio if ratio > 0 else 1.0))\n        target_w = int(round(ratio * scale / 32) * 32)\n        target_h = int(round(scale / 32) * 32)\n    elif dimension_mode == "Custom Dimensions":\n        target_w = int(round(custom_w / 32) * 32)\n        target_h = int(round(custom_h / 32) * 32)\n    else:\n        ASPECT_RATIO_SPECS = {\n            "1:1 (Square)": (1, 1),\n            "16:9 (Widescreen)": (16, 9),\n            "9:16 (Portrait Widescreen)": (9, 16),\n            "21:9 (Ultrawide)": (21, 9),\n            "9:21 (Tall Ultrawide)": (9, 21),\n            "4:3 (Standard)": (4, 3),\n            "3:4 (Portrait Standard)": (3, 4),\n            "3:2 (Photo)": (3, 2),\n            "2:3 (Portrait Photo)": (2, 3),\n            "5:4 (Landscape Tall)": (5, 4),\n            "4:5 (Portrait Tall)": (4, 5),\n            "1:2 (Tall Vertical)": (1, 2),\n            "2:1 (Panoramic)": (2, 1)\n        }\n        w_ratio, h_ratio = ASPECT_RATIO_SPECS.get(preset_ratio_str, (1, 1))\n        total_pixels = megapixels * 1024 * 1024\n        scale = math.sqrt(total_pixels / (w_ratio * h_ratio))\n        target_w = int(round(w_ratio * scale / 32) * 32)\n        target_h = int(round(h_ratio * scale / 32) * 32)\n\n    target_w = max(64, min(2048, target_w))\n    target_h = max(64, min(2048, target_h))\n    return target_w, target_h\n\ndef resolve_lora_selection(selected_val: str, custom_val: str = "") -> str:\n    if not selected_val or selected_val == "None":\n        return None\n    if selected_val.startswith("Custom"):\n        c_name = custom_val.strip().strip(""\'")\n        if not c_name:\n            return None\n        c_name = os.path.basename(c_name)\n        if not (c_name.endswith(".safetensors") or c_name.endswith(".pt") or c_name.endswith(".bin")):\n            c_name += ".safetensors"\n        return c_name\n    return selected_val.strip()\n\ndef download_lora_multisource(url: str, output_dir: str, exact_filename: str, civ_token: str = "", hf_tok: str = ""):\n    os.makedirs(output_dir, exist_ok=True)\n    target_file = os.path.join(output_dir, exact_filename)\n    aria_meta_file = target_file + ".aria2"\n\n    if os.path.exists(target_file) and not os.path.exists(aria_meta_file):\n        size_mb = os.path.getsize(target_file) / (1024 * 1024)\n        if size_mb >= 1.0:\n            return "Existing / Verified", size_mb\n\n    clean_url = url.strip()\n    platform = "Direct Link"\n\n    if "drive.google.com" in clean_url or "docs.google.com" in clean_url:\n        platform = "Google Drive"\n        import gdown\n        gdown.download(clean_url, target_file, fuzzy=True, quiet=False)\n\n    elif "civitai.com" in clean_url:\n        platform = "Civitai"\n        m_ver = re.search(r"modelVersionId=([0-9]+)", clean_url)\n        if m_ver:\n            clean_url = f"https://civitai.com/api/download/models/{m_ver.group(1)}"\n        elif "/models/" in clean_url and "/api/" not in clean_url:\n            m_id = re.search(r"/models/([0-9]+)", clean_url)\n            if m_id:\n                try:\n                    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}\n                    if civ_token.strip():\n                        headers["Authorization"] = f"Bearer {civ_token.strip()}"\n                    res = requests.get(f"https://civitai.com/api/v1/models/{m_id.group(1)}", headers=headers, timeout=6).json()\n                    ver_id = res["modelVersions"][0]["id"]\n                    clean_url = f"https://civitai.com/api/download/models/{ver_id}"\n                except Exception as e:\n                    print(f"Notice: Civitai auto-resolution defaulted to URL: {e}")\n\n        header_flags = ["--header=User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64)"]\n        if civ_token.strip():\n            header_flags.append(f"--header=Authorization: Bearer {civ_token.strip()}")\n            delimiter = "&" if "?" in clean_url else "?"\n            clean_url = f"{clean_url}{delimiter}token={civ_token.strip()}"\n\n        cmd = [\n            "aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M",\n            "--auto-file-renaming=false", "--allow-overwrite=true", "--content-disposition=false",\n            "-d", output_dir, "-o", exact_filename, *header_flags, clean_url\n        ]\n        subprocess.run(cmd)\n\n    elif "huggingface.co" in clean_url:\n        platform = "Hugging Face"\n        clean_url = clean_url.replace("/blob/", "/resolve/")\n        header_flags = []\n        if hf_tok.strip():\n            header_flags.append(f"--header=Authorization: Bearer {hf_tok.strip()}")\n\n        cmd = [\n            "aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M",\n            "--auto-file-renaming=false", "--allow-overwrite=true", "--content-disposition=false",\n            "-d", output_dir, "-o", exact_filename, *header_flags, clean_url\n        ]\n        subprocess.run(cmd)\n\n    else:\n        cmd = [\n            "aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M",\n            "--auto-file-renaming=false", "--allow-overwrite=true", "--content-disposition=false",\n            "-d", output_dir, "-o", exact_filename, clean_url\n        ]\n        subprocess.run(cmd)\n\n    if not os.path.exists(target_file) or os.path.getsize(target_file) < 1024 * 1024:\n        raise RuntimeError(f"Failed to download LoRA from {platform}. Please check URL or file permissions.")\n\n    if os.path.exists(aria_meta_file):\n        os.remove(aria_meta_file)\n\n    size_mb = os.path.getsize(target_file) / (1024 * 1024)\n    return platform, size_mb\n\ndef build_lora_chain(workflow, active_loras, base_model_link, base_clip_link):\n    current_model = base_model_link\n    current_clip = base_clip_link\n    loras_dir = "/content/engine/models/loras" if os.path.exists("/content/engine") else "models/loras"\n    os.makedirs(loras_dir, exist_ok=True)\n\n    PRESET_URLS = {\n        "bfs_head_v1.1_qwen_2.1.safetensors": "https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/resolve/main/bfs_head_v1.1_qwen_2.1.safetensors",\n        "bfs_body_swap_v1.0_qwen_2.1.safetensors": "https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/resolve/main/bfs_body_swap_v1.0_qwen_2.1.safetensors",\n        "Qwen2.1_Anything2RealCharacters.safetensors": "https://huggingface.co/WarmBloodAban/Qwen-Image-2.1-LoRAs/resolve/main/Qwen2.1_Anything2RealCharacters.safetensors",\n        "p_qwen_image_2.1_8step_v0.1.safetensors": "https://huggingface.co/PrunaAI/Pruna-Qwen-Image-2.1/resolve/main/p_qwen_image_2.1_8step_v0.1.safetensors"\n    }\n\n    for idx, lora in enumerate(active_loras):\n        node_id = f"lora_loader_{idx+1}"\n        clean_name = os.path.basename(lora["name"].strip())\n        local_path = os.path.join(loras_dir, clean_name)\n\n        if not os.path.exists(local_path) and clean_name in PRESET_URLS:\n            print(f"📥 Auto-downloading missing LoRA \'{clean_name}\'...")\n            download_lora_multisource(PRESET_URLS[clean_name], loras_dir, clean_name)\n\n        if not os.path.exists(local_path):\n            print(f"⚠️ Warning: LoRA \'{clean_name}\' was not detected in {loras_dir}. Make sure you ran Cell 2 to download it.")\n\n        workflow[node_id] = {\n            "inputs": {\n                "lora_name": clean_name,\n                "strength_model": float(lora.get("strength_model", 1.0)),\n                "strength_clip": float(lora.get("strength_clip", 1.0)),\n                "model": current_model,\n                "clip": current_clip\n            },\n            "class_type": "LoraLoader",\n            "_meta": {"title": f"LoRA Loader #{idx+1} ({clean_name})"}\n        }\n        current_model = [node_id, 0]\n        current_clip = [node_id, 1]\n\n    return current_model, current_clip\n\ndef execute_comfy_generation(workflow, task_name, target_width, target_height, seed, active_loras_summary):\n    ensure_server_alive()\n    purge_backend_memory()\n\n    dims_label = f"{target_width} × {target_height}"\n    display_handle = display(HTML(""), display_id=str(uuid.uuid4()))\n    render_dashboard(display_handle, task_name, "Validating & Submitting Prompt...", "Dispatcher", 0, 0.0, seed, dims_label, active_loras_summary)\n\n    start_t = time.time()\n    prompt_res = queue_prompt(workflow)\n    prompt_id = prompt_res["prompt_id"]\n\n    ws = websocket.WebSocket()\n    ws.connect(f"ws://{server_address}/ws?clientId={client_id}")\n    ws.settimeout(1.0)\n    current_pct = 5\n\n    try:\n        while True:\n            if "server_proc" in globals() and server_proc is not None and server_proc.poll() is not None:\n                render_dashboard(display_handle, task_name, "Server Crashed", "Terminated", current_pct, time.time() - start_t, seed, dims_label, active_loras_summary, error=True)\n                raise RuntimeError("Backend server process terminated unexpectedly.")\n\n            try:\n                out = ws.recv()\n            except websocket.WebSocketTimeoutException:\n                render_dashboard(display_handle, task_name, f"Computing Diffusion Steps ({dims_label})...", "In Progress", current_pct, time.time() - start_t, seed, dims_label, active_loras_summary)\n                continue\n\n            if isinstance(out, str):\n                msg = json.loads(out)\n                msg_type = msg.get("type")\n                elapsed = time.time() - start_t\n\n                if msg_type == "execution_error":\n                    err_info = msg.get("data", {})\n                    err_msg = err_info.get("exception_message", "Execution error")\n                    render_dashboard(display_handle, task_name, f"Error in {err_info.get(\'node_type\')}", err_msg, current_pct, elapsed, seed, dims_label, active_loras_summary, error=True)\n                    raise RuntimeError(f"Pipeline Execution Error: {err_msg}")\n\n                if msg_type == "execution_interrupted":\n                    render_dashboard(display_handle, task_name, "Canceled", "Interrupted", current_pct, elapsed, seed, dims_label, active_loras_summary, error=True)\n                    break\n\n                if msg_type == "executing":\n                    node_id = msg["data"].get("node")\n                    msg_prompt_id = msg["data"].get("prompt_id")\n                    if node_id is None:\n                        if msg_prompt_id is None or msg_prompt_id == prompt_id:\n                            render_dashboard(display_handle, task_name, f"Completed ({dims_label})", "Latent Decoded", 100, elapsed, seed, dims_label, active_loras_summary, done=True)\n                            break\n                    elif str(node_id) in workflow:\n                        node_name = workflow[str(node_id)].get("_meta", {}).get("title", f"Node {node_id}")\n                        render_dashboard(display_handle, task_name, f"Processing ({dims_label})", node_name, current_pct, elapsed, seed, dims_label, active_loras_summary)\n\n                elif msg_type == "progress":\n                    data = msg["data"]\n                    step = data["value"]\n                    total_steps = data["max"]\n                    current_pct = 10 + int((step / total_steps) * 85)\n                    render_dashboard(display_handle, task_name, f"Sampling {step}/{total_steps} ({dims_label})", "KSampler (Euler Simple)", current_pct, elapsed, seed, dims_label, active_loras_summary)\n    finally:\n        try:\n            ws.close()\n        except Exception:\n            pass\n\n    history = None\n    for _ in range(10):\n        try:\n            history_req = urllib.request.urlopen(f"http://{server_address}/history/{prompt_id}")\n            history_data = json.loads(history_req.read().decode("utf-8"))\n            if prompt_id in history_data:\n                history = history_data[prompt_id]\n                break\n        except Exception:\n            pass\n        time.sleep(0.5)\n\n    if not history:\n        raise RuntimeError(f"Failed to fetch execution history for prompt {prompt_id} from backend server.")\n\n    output_images = []\n    for node_id in history.get("outputs", {}):\n        node_output = history["outputs"][node_id]\n        if "images" in node_output:\n            for img_info in node_output["images"]:\n                raw_data = get_image(img_info["filename"], img_info["subfolder"], img_info["type"])\n                output_images.append(Image.open(io.BytesIO(raw_data)))\n\n    purge_backend_memory()\n\n    save_dir = "/content/output"\n    os.makedirs(save_dir, exist_ok=True)\n\n    for i, img in enumerate(output_images):\n        clean_task = task_name.lower().replace(" ", "_").replace("-", "_")\n        out_filename = f"{clean_task}_{seed}_{i+1}.png"\n        out_full_path = os.path.join(save_dir, out_filename)\n        img.save(out_full_path, "PNG")\n\n        display(HTML(f"""\n        <div style="font-family: -apple-system, BlinkMacSystemFont, \'Segoe UI\', Roboto, sans-serif; background: #0d1117; border: 1px solid #30363d; border-radius: 8px; padding: 12px; margin-top: 10px; max-width: 820px;">\n          <div style="display: flex; justify-content: space-between; align-items: center; color: #8b949e; font-size: 11px;">\n            <span>Resolution: <strong style="color: #f0f6fc;">{img.width} × {img.height}</strong> | Render: <strong style="color: #58a6ff;">{time.time() - start_t:.2f}s</strong> | Seed: <strong style="color: #f0f6fc;">{seed}</strong></span>\n            <span>💾 Saved to: <strong style="color: #79c0ff; font-family: monospace;">{out_full_path}</strong></span>\n          </div>\n        </div>\n        """))\n        display(img)\n    return output_images\n'
for _p in ["/content/engine/qwen_studio_utils.py", "/content/qwen_studio_utils.py"]:
    try:
        with open(_p, "w", encoding="utf-8") as f:
            f.write(SHARED_UTILS_CODE)
    except Exception:
        pass

from qwen_studio_utils import *

# 2. Launch or verify server
t_poll = time.time()
ensure_server_alive()

display(HTML(f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background: #0d1117; border: 1px solid #30363d; border-radius: 8px; padding: 12px 16px; max-width: 820px; display: flex; align-items: center; justify-content: space-between;">
  <div style="display: flex; align-items: center; gap: 10px;">
    <span style="display: inline-block; width: 10px; height: 10px; border-radius: 50%; background: #3fb950; box-shadow: 0 0 10px #238636;"></span>
    <span style="color: #f0f6fc; font-weight: 600; font-size: 13px;">Inference Backend Server (Protected against OOM)</span>
    <span style="color: #8b949e; font-size: 11px;">127.0.0.1:8188</span>
  </div>
  <span style="background: rgba(56, 139, 253, 0.15); color: #58a6ff; border: 1px solid rgba(56, 139, 253, 0.4); padding: 2px 10px; border-radius: 12px; font-size: 11px; font-weight: 600;">
    ONLINE ({time.time() - t_poll:.1f}s)
  </span>
</div>
"""))


In [ ]:
# @title 4. Text-to-Image Generation (with Multi-LoRA Support & 2K Native Upscale)
# --- Self-Healing Engine Bootstrap ---
import os, sys
for _p in ["/content/engine", "/content", "."]:
    if _p not in sys.path and os.path.exists(_p):
        sys.path.insert(0, _p)

try:
    from qwen_studio_utils import *
except ImportError:
    if os.path.exists("/content/engine"):
        sys.path.insert(0, "/content/engine")
        from qwen_studio_utils import *
    else:
        raise RuntimeError("⚠️ ComfyUI engine not initialized! Please run 'Cell 1: Environment Setup & Model Synchronization' first.")

import json
import uuid
import random
import math

# --- 1. Prompt Parameters ---
prompt_text = "A high-resolution, full-body photograph of a fashionable 18-year-old female teen walking down a sunlit, high-end shopping street in Paris. She wears an oversized pastel-pink designer bomber jacket draped loosely over her shoulders, paired with a cropped ribbed white seamless top and high-waisted, tailored cream trousers with clean pleats. She carries a mini leather designer handbag in a bold emerald green. Her hair is styled in effortless, voluminous curtain bangs and loose beach waves. She wears sleek oval designer sunglasses, layered dainty gold necklaces, and platform designer sneakers. Bright, natural golden-hour daylight creates soft glows on her skin, clean reflections on boutique windows, and a soft bokeh background of cobblestone streets and luxury storefronts. Captured on a 35mm lens, high contrast, vivid color grading, highly detailed textures." # @param {type:"string"}
negative_prompt = "blurry, low quality, artifacts, distorted, out of focus" # @param {type:"string"}

# --- 2. Dimension Settings & Aspect Ratio Presets ---
dimension_mode = "Preset Aspect Ratio" # @param ["Preset Aspect Ratio", "Custom", "2K High-Res (2048x2048)"]
aspect_ratio = "5:4 (Landscape Tall)" # @param ["1:1 (Square)", "16:9 (Widescreen)", "9:16 (Portrait Widescreen)", "21:9 (Ultrawide)", "9:21 (Tall Ultrawide)", "4:3 (Standard)", "3:4 (Portrait Standard)", "3:2 (Photo)", "2:3 (Portrait Photo)", "5:4 (Landscape Tall)", "4:5 (Portrait Tall)", "1:2 (Tall Vertical)", "2:1 (Panoramic)"]
megapixels = 1.0 # @param {type:"slider", min:0.5, max:4.2, step:0.1}

# Active when dimension_mode is "Custom"
custom_width = 672 # @param {type:"integer"}
custom_height = 1568 # @param {type:"integer"}

# @markdown ---
# @markdown ### ⚡ Speed & VRAM Saver (Fast Draft Mode):
# @markdown Keeps selected aspect ratio but lowers pixel dimensions (runs 2.5–3× faster on Colab T4):
lower_dimensions_for_speed = False # @param {type:"boolean"}
speed_target_resolution = "0.6 Megapixels (Fastest / Low VRAM)" # @param ["0.45 Megapixels (Draft ~576x768)", "0.6 Megapixels (Fastest / Low VRAM)", "0.8 Megapixels (Balanced Speed)"]

# --- 3. Sampling Parameters ---
steps = 18 # @param {type:"integer"}
seed = -1 # @param {type:"integer"}
cfg = 1.0 # @param {type:"number"}

# --- 4. LoRA Controls (Select with Dropdown or Custom) ---
# Primary LoRA
primary_lora = "None" # @param ["None", "p_qwen_image_2.1_8step_v0.1.safetensors", "Qwen2.1_Anything2RealCharacters.safetensors", "bfs_head_v1.1_qwen_2.1.safetensors", "bfs_body_swap_v1.0_qwen_2.1.safetensors", "Custom (Enter filename below)"]
custom_primary_lora = "" # @param {type:"string"}
primary_lora_strength = 1.0 # @param {type:"slider", min:0.0, max:2.0, step:0.05}

# Secondary LoRA (for chaining e.g. 8-Step Turbo Acceleration)
secondary_lora = "None" # @param ["None", "p_qwen_image_2.1_8step_v0.1.safetensors", "Qwen2.1_Anything2RealCharacters.safetensors", "bfs_head_v1.1_qwen_2.1.safetensors", "bfs_body_swap_v1.0_qwen_2.1.safetensors", "Custom (Enter filename below)"]
custom_secondary_lora = "" # @param {type:"string"}
secondary_lora_strength = 1.0 # @param {type:"slider", min:0.0, max:2.0, step:0.05}

auto_optimize_turbo_steps = True # @param {type:"boolean"}

if seed == -1:
    seed = random.randint(1, 10**14)

# Calculate Dimensions & Enforce 32px Alignment for DiT
if lower_dimensions_for_speed:
    speed_budget_map = {
        "0.45 Megapixels (Draft ~576x768)": 0.45,
        "0.6 Megapixels (Fastest / Low VRAM)": 0.60,
        "0.8 Megapixels (Balanced Speed)": 0.80
    }
    fast_mp = speed_budget_map.get(speed_target_resolution, 0.60)
    ASPECT_RATIO_SPECS = {
        "1:1 (Square)": (1, 1),
        "16:9 (Widescreen)": (16, 9),
        "9:16 (Portrait Widescreen)": (9, 16),
        "21:9 (Ultrawide)": (21, 9),
        "9:21 (Tall Ultrawide)": (9, 21),
        "4:3 (Standard)": (4, 3),
        "3:4 (Portrait Standard)": (3, 4),
        "3:2 (Photo)": (3, 2),
        "2:3 (Portrait Photo)": (2, 3),
        "5:4 (Landscape Tall)": (5, 4),
        "4:5 (Portrait Tall)": (4, 5),
        "1:2 (Tall Vertical)": (1, 2),
        "2:1 (Panoramic)": (2, 1)
    }
    w_ratio, h_ratio = ASPECT_RATIO_SPECS.get(aspect_ratio, (1, 1))
    total_pixels = fast_mp * 1024 * 1024
    scale = math.sqrt(total_pixels / (w_ratio * h_ratio))
    target_width = int(round(w_ratio * scale / 32) * 32)
    target_height = int(round(h_ratio * scale / 32) * 32)
    print(f"⚡ Speed Mode Active: Kept {aspect_ratio} aspect ratio. Lowered dimensions to {target_width}×{target_height} (~{fast_mp} MP).")
elif dimension_mode == "Custom":
    target_width = int(round(custom_width / 32) * 32)
    target_height = int(round(custom_height / 32) * 32)
elif dimension_mode == "2K High-Res (2048x2048)":
    target_width = 2048
    target_height = 2048
else:
    ASPECT_RATIO_SPECS = {
        "1:1 (Square)": (1, 1),
        "16:9 (Widescreen)": (16, 9),
        "9:16 (Portrait Widescreen)": (9, 16),
        "21:9 (Ultrawide)": (21, 9),
        "9:21 (Tall Ultrawide)": (9, 21),
        "4:3 (Standard)": (4, 3),
        "3:4 (Portrait Standard)": (3, 4),
        "3:2 (Photo)": (3, 2),
        "2:3 (Portrait Photo)": (2, 3),
        "5:4 (Landscape Tall)": (5, 4),
        "4:5 (Portrait Tall)": (4, 5),
        "1:2 (Tall Vertical)": (1, 2),
        "2:1 (Panoramic)": (2, 1)
    }
    w_ratio, h_ratio = ASPECT_RATIO_SPECS.get(aspect_ratio, (1, 1))
    total_pixels = megapixels * 1024 * 1024
    scale = math.sqrt(total_pixels / (w_ratio * h_ratio))
    target_width = int(round(w_ratio * scale / 32) * 32)
    target_height = int(round(h_ratio * scale / 32) * 32)

target_width = max(64, min(2048, target_width))
target_height = max(64, min(2048, target_height))

# Adaptive encoder resolution for high-res / 2K canvases
pixel_count = target_width * target_height
encoder_resolution = 2048 if (pixel_count >= 1536 * 1024 or max(target_width, target_height) >= 1536) else 1024

# Assemble Active LoRAs from dropdown selections
active_loras = []
resolved_p = resolve_lora_selection(primary_lora, custom_primary_lora)
if resolved_p:
    active_loras.append({"name": resolved_p, "strength_model": primary_lora_strength, "strength_clip": primary_lora_strength})

resolved_s = resolve_lora_selection(secondary_lora, custom_secondary_lora)
if resolved_s:
    active_loras.append({"name": resolved_s, "strength_model": secondary_lora_strength, "strength_clip": secondary_lora_strength})

# Auto-adjust steps if 8-step turbo LoRA is active
if auto_optimize_turbo_steps and any("8step" in (l.get("name") or "").lower() for l in active_loras):
    if steps > 10:
        print(f"🚀 Turbo 8-Step LoRA detected! Automatically adjusting steps: {steps} -> 8 steps for ~8s generation.")
        steps = 8

loras_summary = f"{len(active_loras)} Active" if active_loras else "None"

# Workflow Definition
workflow = {
  "unet_loader": {
    "inputs": {
      "unet_name": "qwen_image_2.1_int8_convrot.safetensors",
      "weight_dtype": "default"
    },
    "class_type": "UNETLoader",
    "_meta": {"title": "Load Diffusion Model"}
  },
  "clip_loader": {
    "inputs": {
      "clip_name": "qwen3vl_8b_int8_convrot.safetensors",
      "type": "qwen_image",
      "device": "default"
    },
    "class_type": "CLIPLoader",
    "_meta": {"title": "Load CLIP"}
  },
  "vae_loader": {
    "inputs": {
      "vae_name": "qwen_image_2.1_vae_bf16.safetensors"
    },
    "class_type": "VAELoader",
    "_meta": {"title": "Load VAE"}
  },
  "empty_latent": {
    "inputs": {
      "width": target_width,
      "height": target_height,
      "batch_size": 1
    },
    "class_type": "EmptyLatentImage",
    "_meta": {"title": "Empty Latent Image"}
  },
  "text_encode": {
    "inputs": {
      "prompt": prompt_text,
      "negative_prompt": negative_prompt,
      "resolution": encoder_resolution,
      "clip": ["clip_loader", 0]
    },
    "class_type": "TextEncodeQwenImage21",
    "_meta": {"title": "Text Encode Qwen Image 2.1"}
  },
  "qwen_cache": {
    "inputs": {
      "device": "auto",
      "dtype": "default",
      "model": ["unet_loader", 0]
    },
    "class_type": "QwenImage21Cache",
    "_meta": {"title": "Qwen Image 2.1 Cache"}
  },
  "ksampler": {
    "inputs": {
      "seed": seed,
      "steps": steps,
      "cfg": cfg,
      "sampler_name": "euler",
      "scheduler": "simple",
      "denoise": 1.0,
      "model": ["qwen_cache", 0],
      "positive": ["text_encode", 0],
      "negative": ["text_encode", 1],
      "latent_image": ["empty_latent", 0]
    },
    "class_type": "KSampler",
    "_meta": {"title": "KSampler"}
  },
  "vae_decode": {
    "inputs": {
      "samples": ["ksampler", 0],
      "vae": ["vae_loader", 0]
    },
    "class_type": "VAEDecode",
    "_meta": {"title": "VAE Decode"}
  },
  "save_image": {
    "inputs": {
      "filename_prefix": "Qwen_image_2.1",
      "format": "png",
      "format.bit_depth": "8-bit",
      "format.input_color_space": "sRGB",
      "images": ["vae_decode", 0]
    },
    "class_type": "SaveImageAdvanced",
    "_meta": {"title": "Save Image (Advanced)"}
  }
}

# Dynamic Multi-LoRA Chaining
if active_loras:
    final_model, final_clip = build_lora_chain(workflow, active_loras, ["unet_loader", 0], ["clip_loader", 0])
    workflow["qwen_cache"]["inputs"]["model"] = final_model
    workflow["text_encode"]["inputs"]["clip"] = final_clip

execute_comfy_generation(workflow, "Text-to-Image Generation", target_width, target_height, seed, loras_summary)


In [ ]:
# @title 5. Single-Image Editing & Style Transformation (1-Image Edit)
# @markdown Edit an existing image, transform its style (e.g. Anime/Drawing into Realistic Human Character via `Anything2RealCharacters`), or add/remove features.
# --- Self-Healing Engine Bootstrap ---
import os, sys
for _p in ["/content/engine", "/content", "."]:
    if _p not in sys.path and os.path.exists(_p):
        sys.path.insert(0, _p)

try:
    from qwen_studio_utils import *
except ImportError:
    if os.path.exists("/content/engine"):
        sys.path.insert(0, "/content/engine")
        from qwen_studio_utils import *
    else:
        raise RuntimeError("⚠️ ComfyUI engine not initialized! Please run 'Cell 1: Environment Setup & Model Synchronization' first.")

import random
import math

# --- 1. Input Image Box ---
# @markdown Provide the input image to edit:
image_source = "upload" # @param ["upload", "path_or_url"]
image_path_or_url = "" # @param {type:"string"}

# --- 2. Prompt & Editing Instructions ---
# @markdown Note: You can reference `<image1>` directly in your prompt to refer to the input image.
prompt_text = "transform <image1> into a photorealistic real human character, natural skin texture, detailed eyes, realistic lighting, 8k uhd" # @param {type:"string"}
negative_prompt = "blurry, low quality, artifacts, distorted, out of focus" # @param {type:"string"}

# --- 3. Output Dimensions & Aspect Ratio Policy ---
# @markdown **Aspect Ratio:** By default, output strictly matches your input image's native aspect ratio unless changed.
dimension_mode = "Match Input Image (Recommended)" # @param ["Match Input Image (Recommended)", "Preset Aspect Ratio", "Custom Dimensions"]
output_megapixels = 1.0 # @param {type:"slider", min:0.5, max:3.0, step:0.1}

# @markdown ---
# @markdown ### ⚡ Speed & VRAM Saver (Aspect-Preserving Fast Mode):
# @markdown When enabled, strictly keeps input image's aspect ratio while lowering dimensions (renders 2.5–3× faster):
lower_dimensions_for_speed = False # @param {type:"boolean"}
speed_target_resolution = "0.6 Megapixels (Fastest / Low VRAM)" # @param ["0.45 Megapixels (Draft ~576x768)", "0.6 Megapixels (Fastest / Low VRAM)", "0.8 Megapixels (Balanced Speed)"]

# Active only when "Preset Aspect Ratio" is selected:
aspect_ratio = "1:1 (Square)" # @param ["1:1 (Square)", "16:9 (Widescreen)", "9:16 (Portrait Widescreen)", "4:3 (Standard)", "3:4 (Portrait Standard)", "3:2 (Photo)", "2:3 (Portrait Photo)", "5:4 (Landscape Tall)", "4:5 (Portrait Tall)"]

# Active only when "Custom Dimensions" is selected:
custom_width = 1024 # @param {type:"integer"}
custom_height = 1024 # @param {type:"integer"}

# --- 4. Sampling Controls ---
steps = 20 # @param {type:"integer"}
seed = -1 # @param {type:"integer"}
cfg = 1.0 # @param {type:"number"}

# --- 5. LoRA Settings (Select from Dropdown or Custom) ---
# Primary LoRA (e.g. Anything2Real for style-to-photorealism)
primary_lora = "Qwen2.1_Anything2RealCharacters.safetensors" # @param ["None", "Qwen2.1_Anything2RealCharacters.safetensors", "p_qwen_image_2.1_8step_v0.1.safetensors", "bfs_head_v1.1_qwen_2.1.safetensors", "bfs_body_swap_v1.0_qwen_2.1.safetensors", "Custom (Enter filename below)"]
custom_primary_lora = "" # @param {type:"string"}
primary_lora_strength = 0.75 # @param {type:"slider", min:0.0, max:2.0, step:0.05}

# Secondary LoRA (e.g. 8-Step Turbo Acceleration)
secondary_lora = "None" # @param ["None", "p_qwen_image_2.1_8step_v0.1.safetensors", "Qwen2.1_Anything2RealCharacters.safetensors", "bfs_head_v1.1_qwen_2.1.safetensors", "bfs_body_swap_v1.0_qwen_2.1.safetensors", "Custom (Enter filename below)"]
custom_secondary_lora = "" # @param {type:"string"}
secondary_lora_strength = 1.0 # @param {type:"slider", min:0.0, max:2.0, step:0.05}

auto_optimize_turbo_steps = True # @param {type:"boolean"}

if seed == -1:
    seed = random.randint(1, 10**14)

# 1. Load original input image & detect natural aspect ratio
raw_input_img = load_raw_image(source_mode=image_source, path_or_url=image_path_or_url, default_filename="edit_input_image.png")
orig_w, orig_h = raw_input_img.size

# Display sleek input preview
display_comparison_preview("Input Image to Edit", {"Input Image (<image1>)": raw_input_img})

# 2. Calculate canvas dimensions (strictly preserving input aspect ratio)
if lower_dimensions_for_speed:
    speed_budget_map = {
        "0.45 Megapixels (Draft ~576x768)": 0.45,
        "0.6 Megapixels (Fastest / Low VRAM)": 0.60,
        "0.8 Megapixels (Balanced Speed)": 0.80
    }
    fast_mp = speed_budget_map.get(speed_target_resolution, 0.60)
    ratio = orig_w / orig_h
    total_px = fast_mp * 1024 * 1024
    scale = math.sqrt(total_px / (ratio if ratio > 0 else 1.0))
    target_width = max(64, min(2048, int(round(ratio * scale / 32) * 32)))
    target_height = max(64, min(2048, int(round(scale / 32) * 32)))
    print(f"⚡ Speed Mode Active: Strictly preserved input aspect ratio ({orig_w}:{orig_h} -> {orig_w/orig_h:.2f}). Lowered canvas to {target_width}×{target_height} (~{fast_mp} MP).")
else:
    target_width, target_height = calculate_target_dimensions(
        dimension_mode=dimension_mode,
        orig_w=orig_w,
        orig_h=orig_h,
        megapixels=output_megapixels,
        preset_ratio_str=aspect_ratio,
        custom_w=custom_width,
        custom_h=custom_height
    )

# 3. Save aligned image to ComfyUI input directory
clean_filename, _, _ = save_processed_image(raw_input_img, "edit_input_image.png", target_dims=(target_width, target_height))

# Assemble Active LoRAs from dropdown selections
active_loras = []
resolved_p = resolve_lora_selection(primary_lora, custom_primary_lora)
if resolved_p:
    active_loras.append({"name": resolved_p, "strength_model": primary_lora_strength, "strength_clip": primary_lora_strength})

resolved_s = resolve_lora_selection(secondary_lora, custom_secondary_lora)
if resolved_s:
    active_loras.append({"name": resolved_s, "strength_model": secondary_lora_strength, "strength_clip": secondary_lora_strength})

# Auto-adjust steps if 8-step turbo LoRA is active
if auto_optimize_turbo_steps and any("8step" in (l.get("name") or "").lower() for l in active_loras):
    if steps > 10:
        print(f"🚀 Turbo 8-Step LoRA detected! Automatically adjusting steps: {steps} -> 8 steps for ~8s generation.")
        steps = 8

loras_summary = f"{len(active_loras)} Active" if active_loras else "None"

# Workflow Definition for Image Edit
workflow = {
  "unet_loader": {
    "inputs": {
      "unet_name": "qwen_image_2.1_int8_convrot.safetensors",
      "weight_dtype": "default"
    },
    "class_type": "UNETLoader",
    "_meta": {"title": "Load Diffusion Model"}
  },
  "clip_loader": {
    "inputs": {
      "clip_name": "qwen3vl_8b_int8_convrot.safetensors",
      "type": "qwen_image",
      "device": "default"
    },
    "class_type": "CLIPLoader",
    "_meta": {"title": "Load CLIP"}
  },
  "vae_loader": {
    "inputs": {
      "vae_name": "qwen_image_2.1_vae_bf16.safetensors"
    },
    "class_type": "VAELoader",
    "_meta": {"title": "Load VAE"}
  },
  "load_input_image": {
    "inputs": {
      "image": clean_filename
    },
    "class_type": "LoadImage",
    "_meta": {"title": "Load Input Image"}
  },
  "text_encode_edit": {
    "inputs": {
      "prompt": prompt_text,
      "negative_prompt": negative_prompt,
      "resolution": 0,
      "clip": ["clip_loader", 0],
      "vae": ["vae_loader", 0],
      "images.image_1": ["load_input_image", 0]
    },
    "class_type": "TextEncodeQwenImage21",
    "_meta": {"title": "Text Encode Qwen Image 2.1 (Edit)"}
  },
  "empty_latent": {
    "inputs": {
      "width": target_width,
      "height": target_height,
      "batch_size": 1
    },
    "class_type": "EmptyLatentImage",
    "_meta": {"title": "Empty Latent Image"}
  },
  "qwen_cache": {
    "inputs": {
      "device": "auto",
      "dtype": "default",
      "model": ["unet_loader", 0]
    },
    "class_type": "QwenImage21Cache",
    "_meta": {"title": "Qwen Image 2.1 Cache"}
  },
  "ksampler": {
    "inputs": {
      "seed": seed,
      "steps": steps,
      "cfg": cfg,
      "sampler_name": "euler",
      "scheduler": "simple",
      "denoise": 1.0,
      "model": ["qwen_cache", 0],
      "positive": ["text_encode_edit", 0],
      "negative": ["text_encode_edit", 1],
      "latent_image": ["empty_latent", 0]
    },
    "class_type": "KSampler",
    "_meta": {"title": "KSampler"}
  },
  "vae_decode": {
    "inputs": {
      "samples": ["ksampler", 0],
      "vae": ["vae_loader", 0]
    },
    "class_type": "VAEDecode",
    "_meta": {"title": "VAE Decode"}
  },
  "save_image": {
    "inputs": {
      "filename_prefix": "Qwen_edit",
      "format": "png",
      "format.bit_depth": "8-bit",
      "format.input_color_space": "sRGB",
      "images": ["vae_decode", 0]
    },
    "class_type": "SaveImageAdvanced",
    "_meta": {"title": "Save Edited Image"}
  }
}

# Dynamic Multi-LoRA Chaining
if active_loras:
    final_model, final_clip = build_lora_chain(workflow, active_loras, ["unet_loader", 0], ["clip_loader", 0])
    workflow["qwen_cache"]["inputs"]["model"] = final_model
    workflow["text_encode_edit"]["inputs"]["clip"] = final_clip

execute_comfy_generation(workflow, "Single-Image Editing", target_width, target_height, seed, loras_summary)


In [ ]:
# @title 6. Dedicated Head Swap (Picture 1 Target Scene + Picture 2 Reference Face)
# @markdown ### 📸 Image Role Guide:
# @markdown - **Picture 1 (`<image1>`) [TARGET BASE SCENE]:** The person whose body, pose, clothing, background, lighting, and camera angle are strictly preserved.
# @markdown - **Picture 2 (`<image2>`) [REFERENCE HEAD / IDENTITY]:** The person whose facial features, eyes, nose, lips, hair, and identity are transferred into Picture 1.
# --- Self-Healing Engine Bootstrap ---
import os, sys
for _p in ["/content/engine", "/content", "."]:
    if _p not in sys.path and os.path.exists(_p):
        sys.path.insert(0, _p)

try:
    from qwen_studio_utils import *
except ImportError:
    if os.path.exists("/content/engine"):
        sys.path.insert(0, "/content/engine")
        from qwen_studio_utils import *
    else:
        raise RuntimeError("⚠️ ComfyUI engine not initialized! Please run 'Cell 1: Environment Setup & Model Synchronization' first.")

import random
import math

# --- 1. Picture 1: Target Base Scene ---
picture1_source = "upload" # @param ["upload", "path_or_url"]
picture1_path_or_url = "" # @param {type:"string"}

# --- 2. Picture 2: Reference Head / Face ---
picture2_source = "upload" # @param ["upload", "path_or_url"]
picture2_path_or_url = "" # @param {type:"string"}

# --- 3. Custom Prompt (Optional) ---
# @markdown Optional custom prompt to attach after the default head swap prompt (e.g. "smiling gently, sunny weather"):
custom_prompt = "" # @param {type:"string"}
negative_prompt = "" # @param {type:"string"}

# --- 4. Quality & Output Dimensions ---
# @markdown **Aspect Ratio:** In BFS Head Swap, output aspect ratio strictly matches Target Scene (Picture 1) unless overridden:
dimension_mode = "Match Target Image (Picture 1)" # @param ["Match Target Image (Picture 1)", "Preset Aspect Ratio", "Custom Dimensions"]
scene_megapixels = 1.5 # @param {type:"slider", min:0.5, max:3.0, step:0.1}

# @markdown ---
# @markdown ### ⚡ Speed & VRAM Saver (Aspect-Preserving Fast Mode):
# @markdown When enabled, strictly keeps Picture 1 (Target Scene) aspect ratio while lowering dimensions (renders 2.5–3× faster):
lower_dimensions_for_speed = False # @param {type:"boolean"}
speed_target_resolution = "0.6 Megapixels (Fastest / Low VRAM)" # @param ["0.45 Megapixels (Draft ~576x768)", "0.6 Megapixels (Fastest / Low VRAM)", "0.8 Megapixels (Balanced Speed)"]

# Active only when "Preset Aspect Ratio" is selected:
aspect_ratio = "1:1 (Square)" # @param ["1:1 (Square)", "16:9 (Widescreen)", "9:16 (Portrait Widescreen)", "4:3 (Standard)", "3:4 (Portrait Standard)", "3:2 (Photo)", "2:3 (Portrait Photo)", "5:4 (Landscape Tall)", "4:5 (Portrait Tall)"]

# Active only when "Custom Dimensions" is selected:
custom_width = 1024 # @param {type:"integer"}
custom_height = 1024 # @param {type:"integer"}

# --- 5. Sampling Parameters ---
steps = 22 # @param {type:"integer"}
seed = -1 # @param {type:"integer"}
cfg = 1.0 # @param {type:"number"}

# --- 6. LoRA Controls (Select with Dropdown or Custom) ---
# Primary LoRA: BFS Head Swap V1.1
primary_lora = "bfs_head_v1.1_qwen_2.1.safetensors" # @param ["None", "bfs_head_v1.1_qwen_2.1.safetensors", "p_qwen_image_2.1_8step_v0.1.safetensors", "bfs_body_swap_v1.0_qwen_2.1.safetensors", "Qwen2.1_Anything2RealCharacters.safetensors", "Custom (Enter filename below)"]
custom_primary_lora = "" # @param {type:"string"}
primary_lora_strength = 1.0 # @param {type:"slider", min:0.0, max:2.0, step:0.05}

# Secondary LoRA: Optional Turbo Acceleration (8-Step)
secondary_lora = "None" # @param ["None", "p_qwen_image_2.1_8step_v0.1.safetensors", "bfs_head_v1.1_qwen_2.1.safetensors", "bfs_body_swap_v1.0_qwen_2.1.safetensors", "Qwen2.1_Anything2RealCharacters.safetensors", "Custom (Enter filename below)"]
custom_secondary_lora = "" # @param {type:"string"}
secondary_lora_strength = 1.0 # @param {type:"slider", min:0.0, max:2.0, step:0.05}

auto_optimize_turbo_steps = True # @param {type:"boolean"}

if seed == -1:
    seed = random.randint(1, 10**14)

# Default Head Swap Prompt (Strict Preservation Contract)
DEFAULT_HEAD_SWAP_PROMPT = """head_swap: start with Picture 1 as the base image, keeping its lighting, environment, and background. Remove the head from Picture 1 completely and replace it with the head from Picture 2.
FROM PICTURE 1 (strictly preserve):
- Scene: lighting conditions, shadows, highlights, color temperature, environment, background
- Head positioning: exact rotation angle, tilt, direction the head is facing
- Expression: facial expression, micro-expressions, eye gaze direction, mouth position, emotion
FROM PICTURE 2 (strictly preserve identity):
- Facial structure: face shape, bone structure, jawline, chin
- All facial features: eye color, eye shape, nose structure, lip shape and fullness, eyebrows
- Hair: color, style, texture, hairline
- Skin: texture, tone, complexion
The replaced head must seamlessly match Picture 1's lighting and expression while maintaining the complete identity from Picture 2. High quality, photorealistic, sharp details, 4k."""

# Combine default + custom prompt
if custom_prompt.strip():
    final_prompt = f"{DEFAULT_HEAD_SWAP_PROMPT}\n{custom_prompt.strip()}"
else:
    final_prompt = DEFAULT_HEAD_SWAP_PROMPT

# 1. Load Picture 1 (Target Scene) & Picture 2 (Reference Head)
raw_pic1 = load_raw_image(source_mode=picture1_source, path_or_url=picture1_path_or_url, default_filename="head_pic1_scene.png")
pic1_orig_w, pic1_orig_h = raw_pic1.size

raw_pic2 = load_raw_image(source_mode=picture2_source, path_or_url=picture2_path_or_url, default_filename="head_pic2_ref.png")

# Display sleek input preview
display_comparison_preview("Head Swap Input References", {
    "Picture 1: Target Scene (Base)": raw_pic1,
    "Picture 2: Reference Head (Identity)": raw_pic2
})

# 2. Calculate canvas matching Picture 1's aspect ratio
if lower_dimensions_for_speed:
    speed_budget_map = {
        "0.45 Megapixels (Draft ~576x768)": 0.45,
        "0.6 Megapixels (Fastest / Low VRAM)": 0.60,
        "0.8 Megapixels (Balanced Speed)": 0.80
    }
    fast_mp = speed_budget_map.get(speed_target_resolution, 0.60)
    ratio = pic1_orig_w / pic1_orig_h
    total_px = fast_mp * 1024 * 1024
    scale = math.sqrt(total_px / (ratio if ratio > 0 else 1.0))
    target_width = max(64, min(2048, int(round(ratio * scale / 32) * 32)))
    target_height = max(64, min(2048, int(round(scale / 32) * 32)))
    print(f"⚡ Speed Mode Active: Strictly preserved Picture 1 (Target Scene) aspect ratio ({pic1_orig_w}:{pic1_orig_h} -> {pic1_orig_w/pic1_orig_h:.2f}). Lowered canvas to {target_width}×{target_height} (~{fast_mp} MP).")
else:
    target_width, target_height = calculate_target_dimensions(
        dimension_mode=dimension_mode,
        orig_w=pic1_orig_w,
        orig_h=pic1_orig_h,
        megapixels=scene_megapixels,
        preset_ratio_str=aspect_ratio,
        custom_w=custom_width,
        custom_h=custom_height
    )

# 3. Save Picture 1 aligned to canvas dimensions
pic1_file, _, _ = save_processed_image(raw_pic1, "head_pic1_scene.png", target_dims=(target_width, target_height))

# 4. Save Picture 2 (Reference Head) scaled to ~1.0 MP preserving its own aspect ratio
pic2_file, _, _ = save_processed_image(raw_pic2, "head_pic2_ref.png", target_area_mp=1.0)

# Assemble Active LoRAs from dropdown selections
active_loras = []
resolved_p = resolve_lora_selection(primary_lora, custom_primary_lora)
if resolved_p:
    active_loras.append({"name": resolved_p, "strength_model": primary_lora_strength, "strength_clip": primary_lora_strength})

resolved_s = resolve_lora_selection(secondary_lora, custom_secondary_lora)
if resolved_s:
    active_loras.append({"name": resolved_s, "strength_model": secondary_lora_strength, "strength_clip": secondary_lora_strength})

# Auto-adjust steps if 8-step turbo LoRA is active
if auto_optimize_turbo_steps and any("8step" in (l.get("name") or "").lower() for l in active_loras):
    if steps > 10:
        print(f"🚀 Turbo 8-Step LoRA detected! Automatically adjusting steps: {steps} -> 8 steps for ~8s generation.")
        steps = 8

loras_summary = f"{len(active_loras)} Active" if active_loras else "None"

# Workflow Definition for Head Swap
workflow = {
  "unet_loader": {
    "inputs": {
      "unet_name": "qwen_image_2.1_int8_convrot.safetensors",
      "weight_dtype": "default"
    },
    "class_type": "UNETLoader",
    "_meta": {"title": "Load Diffusion Model"}
  },
  "clip_loader": {
    "inputs": {
      "clip_name": "qwen3vl_8b_int8_convrot.safetensors",
      "type": "qwen_image",
      "device": "default"
    },
    "class_type": "CLIPLoader",
    "_meta": {"title": "Load CLIP"}
  },
  "vae_loader": {
    "inputs": {
      "vae_name": "qwen_image_2.1_vae_bf16.safetensors"
    },
    "class_type": "VAELoader",
    "_meta": {"title": "Load VAE"}
  },
  "load_pic1": {
    "inputs": {
      "image": pic1_file
    },
    "class_type": "LoadImage",
    "_meta": {"title": "Picture 1: Base Scene / Target"}
  },
  "load_pic2": {
    "inputs": {
      "image": pic2_file
    },
    "class_type": "LoadImage",
    "_meta": {"title": "Picture 2: Reference Head / Identity"}
  },
  "text_encode_head": {
    "inputs": {
      "prompt": final_prompt,
      "negative_prompt": negative_prompt,
      "resolution": 0,
      "clip": ["clip_loader", 0],
      "vae": ["vae_loader", 0],
      "images.image_1": ["load_pic1", 0],
      "images.image_2": ["load_pic2", 0]
    },
    "class_type": "TextEncodeQwenImage21",
    "_meta": {"title": "Text Encode Qwen Image 2.1 (Head Swap)"}
  },
  "empty_latent": {
    "inputs": {
      "width": target_width,
      "height": target_height,
      "batch_size": 1
    },
    "class_type": "EmptyLatentImage",
    "_meta": {"title": "Empty Latent Image"}
  },
  "qwen_cache": {
    "inputs": {
      "device": "auto",
      "dtype": "default",
      "model": ["unet_loader", 0]
    },
    "class_type": "QwenImage21Cache",
    "_meta": {"title": "Qwen Image 2.1 Cache"}
  },
  "ksampler": {
    "inputs": {
      "seed": seed,
      "steps": steps,
      "cfg": cfg,
      "sampler_name": "euler",
      "scheduler": "simple",
      "denoise": 1.0,
      "model": ["qwen_cache", 0],
      "positive": ["text_encode_head", 0],
      "negative": ["text_encode_head", 1],
      "latent_image": ["empty_latent", 0]
    },
    "class_type": "KSampler",
    "_meta": {"title": "KSampler"}
  },
  "vae_decode": {
    "inputs": {
      "samples": ["ksampler", 0],
      "vae": ["vae_loader", 0]
    },
    "class_type": "VAEDecode",
    "_meta": {"title": "VAE Decode"}
  },
  "save_image": {
    "inputs": {
      "filename_prefix": "Qwen_head_swap",
      "format": "png",
      "format.bit_depth": "8-bit",
      "format.input_color_space": "sRGB",
      "images": ["vae_decode", 0]
    },
    "class_type": "SaveImageAdvanced",
    "_meta": {"title": "Save Head Swap Image"}
  }
}

# Dynamic Multi-LoRA Chaining
if active_loras:
    final_model, final_clip = build_lora_chain(workflow, active_loras, ["unet_loader", 0], ["clip_loader", 0])
    workflow["qwen_cache"]["inputs"]["model"] = final_model
    workflow["text_encode_head"]["inputs"]["clip"] = final_clip

execute_comfy_generation(workflow, "Dedicated Head Swap", target_width, target_height, seed, loras_summary)


In [ ]:
# @title 7. Dedicated Body Swap (Image 1 Scene + Image 2 Reference Body & Clothes)
# @markdown ### 🧍 Body Swap Assignment Guide:
# @markdown - **Image 1 (`<image1>`) [SCENE BASE]:** The scene. Its pose, framing, lighting, and background are kept.
# @markdown - **Image 2 (`<image2>`) [REFERENCE PERSON]:** The reference person, whose face, clothing, and body proportions are transferred.
# --- Self-Healing Engine Bootstrap ---
import os, sys
for _p in ["/content/engine", "/content", "."]:
    if _p not in sys.path and os.path.exists(_p):
        sys.path.insert(0, _p)

try:
    from qwen_studio_utils import *
except ImportError:
    if os.path.exists("/content/engine"):
        sys.path.insert(0, "/content/engine")
        from qwen_studio_utils import *
    else:
        raise RuntimeError("⚠️ ComfyUI engine not initialized! Please run 'Cell 1: Environment Setup & Model Synchronization' first.")

import random
import math

# --- 1. Image 1: Scene (Base) ---
image1_source = "upload" # @param ["upload", "path_or_url"]
image1_path_or_url = "" # @param {type:"string"}

# --- 2. Image 2: Reference Person (Clothing & Body) ---
image2_source = "upload" # @param ["upload", "path_or_url"]
image2_path_or_url = "" # @param {type:"string"}

# --- 3. Custom Prompt (Optional) ---
# @markdown Optional custom prompt to attach after the default body swap prompt:
custom_prompt = "" # @param {type:"string"}
negative_prompt = "" # @param {type:"string"}

# --- 4. Quality & Output Dimensions ---
# @markdown **Aspect Ratio:** In BFS Body Swap, output aspect ratio strictly matches Target Scene (Image 1) unless overridden:
dimension_mode = "Match Target Image (Image 1)" # @param ["Match Target Image (Image 1)", "Preset Aspect Ratio", "Custom Dimensions"]
scene_megapixels = 1.5 # @param {type:"slider", min:0.5, max:3.0, step:0.1}

# @markdown ---
# @markdown ### ⚡ Speed & VRAM Saver (Aspect-Preserving Fast Mode):
# @markdown When enabled, strictly keeps Image 1 (Target Scene) aspect ratio while lowering dimensions (renders 2.5–3× faster):
lower_dimensions_for_speed = False # @param {type:"boolean"}
speed_target_resolution = "0.6 Megapixels (Fastest / Low VRAM)" # @param ["0.45 Megapixels (Draft ~576x768)", "0.6 Megapixels (Fastest / Low VRAM)", "0.8 Megapixels (Balanced Speed)"]

# Active only when "Preset Aspect Ratio" is selected:
aspect_ratio = "1:1 (Square)" # @param ["1:1 (Square)", "16:9 (Widescreen)", "9:16 (Portrait Widescreen)", "4:3 (Standard)", "3:4 (Portrait Standard)", "3:2 (Photo)", "2:3 (Portrait Photo)", "5:4 (Landscape Tall)", "4:5 (Portrait Tall)"]

# Active only when "Custom Dimensions" is selected:
custom_width = 1024 # @param {type:"integer"}
custom_height = 1024 # @param {type:"integer"}

# --- 5. Sampling Parameters ---
steps = 22 # @param {type:"integer"}
seed = -1 # @param {type:"integer"}
cfg = 1.0 # @param {type:"number"}

# --- 6. LoRA Controls (Select with Dropdown or Custom) ---
# Primary LoRA: BFS Body Swap V1.0
primary_lora = "bfs_body_swap_v1.0_qwen_2.1.safetensors" # @param ["None", "bfs_body_swap_v1.0_qwen_2.1.safetensors", "p_qwen_image_2.1_8step_v0.1.safetensors", "bfs_head_v1.1_qwen_2.1.safetensors", "Qwen2.1_Anything2RealCharacters.safetensors", "Custom (Enter filename below)"]
custom_primary_lora = "" # @param {type:"string"}
primary_lora_strength = 1.0 # @param {type:"slider", min:0.0, max:2.0, step:0.05}

# Secondary LoRA: Optional Turbo Acceleration (8-Step)
secondary_lora = "None" # @param ["None", "p_qwen_image_2.1_8step_v0.1.safetensors", "bfs_body_swap_v1.0_qwen_2.1.safetensors", "bfs_head_v1.1_qwen_2.1.safetensors", "Qwen2.1_Anything2RealCharacters.safetensors", "Custom (Enter filename below)"]
custom_secondary_lora = "" # @param {type:"string"}
secondary_lora_strength = 1.0 # @param {type:"slider", min:0.0, max:2.0, step:0.05}

auto_optimize_turbo_steps = True # @param {type:"boolean"}

if seed == -1:
    seed = random.randint(1, 10**14)

# Default Body Swap Prompt (Strict Preservation Contract)
DEFAULT_BODY_SWAP_PROMPT = "body_swap: start with <image1> as the base image, keeping its lighting, environment, and background. replace the body from <image1> with the body from <image2>, strictly preserving the clothing, body shape and proportions from <image2>. copy the pose, direction of the eye, head rotation, micro expressions from <image1>"

# Combine default + custom prompt (default + custom prompt as requested)
if custom_prompt.strip():
    final_prompt = f"{DEFAULT_BODY_SWAP_PROMPT}, {custom_prompt.strip()}"
else:
    final_prompt = DEFAULT_BODY_SWAP_PROMPT

# 1. Load Image 1 (Target Scene) & Image 2 (Reference Person)
raw_img1 = load_raw_image(source_mode=image1_source, path_or_url=image1_path_or_url, default_filename="body_img1_scene.png")
img1_orig_w, img1_orig_h = raw_img1.size

raw_img2 = load_raw_image(source_mode=image2_source, path_or_url=image2_path_or_url, default_filename="body_img2_ref.png")

# Display sleek input preview
display_comparison_preview("Body Swap Input References", {
    "Image 1: Scene Base (Preserved)": raw_img1,
    "Image 2: Reference Person (Transferred)": raw_img2
})

# 2. Calculate canvas matching Image 1's aspect ratio
if lower_dimensions_for_speed:
    speed_budget_map = {
        "0.45 Megapixels (Draft ~576x768)": 0.45,
        "0.6 Megapixels (Fastest / Low VRAM)": 0.60,
        "0.8 Megapixels (Balanced Speed)": 0.80
    }
    fast_mp = speed_budget_map.get(speed_target_resolution, 0.60)
    ratio = img1_orig_w / img1_orig_h
    total_px = fast_mp * 1024 * 1024
    scale = math.sqrt(total_px / (ratio if ratio > 0 else 1.0))
    target_width = max(64, min(2048, int(round(ratio * scale / 32) * 32)))
    target_height = max(64, min(2048, int(round(scale / 32) * 32)))
    print(f"⚡ Speed Mode Active: Strictly preserved Image 1 (Target Scene) aspect ratio ({img1_orig_w}:{img1_orig_h} -> {img1_orig_w/img1_orig_h:.2f}). Lowered canvas to {target_width}×{target_height} (~{fast_mp} MP).")
else:
    target_width, target_height = calculate_target_dimensions(
        dimension_mode=dimension_mode,
        orig_w=img1_orig_w,
        orig_h=img1_orig_h,
        megapixels=scene_megapixels,
        preset_ratio_str=aspect_ratio,
        custom_w=custom_width,
        custom_h=custom_height
    )

# 3. Save Image 1 aligned to canvas dimensions
img1_file, _, _ = save_processed_image(raw_img1, "body_img1_scene.png", target_dims=(target_width, target_height))

# 4. Save Image 2 (Reference Person)
# CRITICAL RESEARCH INSIGHT: Scaling reference to ~0.59 MP (768x768 area) prevents reference background
# bleed and guarantees the scene background survives without overriding it.
img2_file, _, _ = save_processed_image(raw_img2, "body_img2_ref.png", target_area_mp=0.59)

# Assemble Active LoRAs from dropdown selections
active_loras = []
resolved_p = resolve_lora_selection(primary_lora, custom_primary_lora)
if resolved_p:
    active_loras.append({"name": resolved_p, "strength_model": primary_lora_strength, "strength_clip": primary_lora_strength})

resolved_s = resolve_lora_selection(secondary_lora, custom_secondary_lora)
if resolved_s:
    active_loras.append({"name": resolved_s, "strength_model": secondary_lora_strength, "strength_clip": secondary_lora_strength})

# Auto-adjust steps if 8-step turbo LoRA is active
if auto_optimize_turbo_steps and any("8step" in (l.get("name") or "").lower() for l in active_loras):
    if steps > 10:
        print(f"🚀 Turbo 8-Step LoRA detected! Automatically adjusting steps: {steps} -> 8 steps for ~8s generation.")
        steps = 8

loras_summary = f"{len(active_loras)} Active" if active_loras else "None"

# Workflow Definition for Body Swap
workflow = {
  "unet_loader": {
    "inputs": {
      "unet_name": "qwen_image_2.1_int8_convrot.safetensors",
      "weight_dtype": "default"
    },
    "class_type": "UNETLoader",
    "_meta": {"title": "Load Diffusion Model"}
  },
  "clip_loader": {
    "inputs": {
      "clip_name": "qwen3vl_8b_int8_convrot.safetensors",
      "type": "qwen_image",
      "device": "default"
    },
    "class_type": "CLIPLoader",
    "_meta": {"title": "Load CLIP"}
  },
  "vae_loader": {
    "inputs": {
      "vae_name": "qwen_image_2.1_vae_bf16.safetensors"
    },
    "class_type": "VAELoader",
    "_meta": {"title": "Load VAE"}
  },
  "load_img1": {
    "inputs": {
      "image": img1_file
    },
    "class_type": "LoadImage",
    "_meta": {"title": "Image 1: Scene Base"}
  },
  "load_img2": {
    "inputs": {
      "image": img2_file
    },
    "class_type": "LoadImage",
    "_meta": {"title": "Image 2: Reference Person"}
  },
  "text_encode_body": {
    "inputs": {
      "prompt": final_prompt,
      "negative_prompt": negative_prompt,
      "resolution": 0,
      "clip": ["clip_loader", 0],
      "vae": ["vae_loader", 0],
      "images.image_1": ["load_img1", 0],
      "images.image_2": ["load_img2", 0]
    },
    "class_type": "TextEncodeQwenImage21",
    "_meta": {"title": "Text Encode Qwen Image 2.1 (Body Swap)"}
  },
  "empty_latent": {
    "inputs": {
      "width": target_width,
      "height": target_height,
      "batch_size": 1
    },
    "class_type": "EmptyLatentImage",
    "_meta": {"title": "Empty Latent Image"}
  },
  "qwen_cache": {
    "inputs": {
      "device": "auto",
      "dtype": "default",
      "model": ["unet_loader", 0]
    },
    "class_type": "QwenImage21Cache",
    "_meta": {"title": "Qwen Image 2.1 Cache"}
  },
  "ksampler": {
    "inputs": {
      "seed": seed,
      "steps": steps,
      "cfg": cfg,
      "sampler_name": "euler",
      "scheduler": "simple",
      "denoise": 1.0,
      "model": ["qwen_cache", 0],
      "positive": ["text_encode_body", 0],
      "negative": ["text_encode_body", 1],
      "latent_image": ["empty_latent", 0]
    },
    "class_type": "KSampler",
    "_meta": {"title": "KSampler"}
  },
  "vae_decode": {
    "inputs": {
      "samples": ["ksampler", 0],
      "vae": ["vae_loader", 0]
    },
    "class_type": "VAEDecode",
    "_meta": {"title": "VAE Decode"}
  },
  "save_image": {
    "inputs": {
      "filename_prefix": "Qwen_body_swap",
      "format": "png",
      "format.bit_depth": "8-bit",
      "format.input_color_space": "sRGB",
      "images": ["vae_decode", 0]
    },
    "class_type": "SaveImageAdvanced",
    "_meta": {"title": "Save Body Swap Image"}
  }
}

# Dynamic Multi-LoRA Chaining
if active_loras:
    final_model, final_clip = build_lora_chain(workflow, active_loras, ["unet_loader", 0], ["clip_loader", 0])
    workflow["qwen_cache"]["inputs"]["model"] = final_model
    workflow["text_encode_body"]["inputs"]["clip"] = final_clip

execute_comfy_generation(workflow, "Dedicated Body Swap", target_width, target_height, seed, loras_summary)
